# *PIP install dependencies*

In [1]:
!pip install wfdb h5py -q

# *Google Colab mount/remount*

In [ ]:
from google.colab import drive

try:
    print("Unmounting...")
    drive.flush_and_unmount()
except:
    pass

print("Remounting...")
drive.mount('/content/drive', force_remount=True)
print("Drive Remounted Successfully.")

# Exploratory Data Analysis

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import wfdb
import h5py
import time
from scipy import signal

# --- GLOBAL CONFIGURATION ---
class Config:
    SHDB_PATH = "/content/drive/MyDrive/sp datasets/shdb-af-a-japanese-holter-ecg-database-of-atrial-fibrillation-1.0.1"
    IRIDIA_PATH = "/content/drive/MyDrive/sp datasets/iridia-af-records-v1.0.1"

    TARGET_FS = 125
    WINDOW_SIZE = 30 * 125  # 3750 samples
    AF_THRESHOLD = 0.5
    SAFE_BUFFER = 6000      # Skip start of files

    # Padding
    PAD_SEC = 5
    PAD_SAMPLES = PAD_SEC * 125

    # Filter
    LOW = 0.5 / (0.5 * 125); HIGH = 40.0 / (0.5 * 125)
    B, A = signal.butter(6, [LOW, HIGH], btype='band')

cfg = Config()

# --- PREPROCESSING FUNCTION ---
def filter_with_padding(raw_padded_chunk):
    seg = signal.filtfilt(cfg.B, cfg.A, raw_padded_chunk)
    if len(seg) > 2 * cfg.PAD_SAMPLES:
        valid_seg = seg[cfg.PAD_SAMPLES : -cfg.PAD_SAMPLES]
    else: valid_seg = seg
    mu, sigma = np.mean(valid_seg), np.std(valid_seg)
    if sigma == 0: return valid_seg
    return (valid_seg - mu) / sigma

# --- DRY RUN LOGIC CHECK ---
print(">>> 🛠️ INITIALIZING PIPELINE (STRICT ATR FILTER)...")
stats = {'SHDB': {'Scanned': 0, 'Skipped_NoATR': 0}, 'IRIDIA': {'Scanned': 0}}

# Quick SHDB Check
hea_files = glob.glob(os.path.join(cfg.SHDB_PATH, "*.hea"))
for hea in hea_files[:20]: # Check first 20
    path = os.path.splitext(hea)[0]

    # --- EXPLICIT FIX: Check if ATR exists ---
    if not os.path.exists(path + '.atr'):
        stats['SHDB']['Skipped_NoATR'] += 1
        continue

    try:
        ann = wfdb.rdann(path, 'atr')
        stats['SHDB']['Scanned'] += 1
    except: pass

print(f"✅ Logic Ready.")
print(f"   SHDB Scanned: {stats['SHDB']['Scanned']} | Skipped (No ATR): {stats['SHDB']['Skipped_NoATR']}")

## Visualization of Raw ECG Segments

### SHDB-AF

In [ ]:
import os
import glob
import numpy as np
import wfdb
import matplotlib.pyplot as plt

SHDB_PATH = "/content/drive/MyDrive/sp datasets/shdb-af-a-japanese-holter-ecg-database-of-atrial-fibrillation-1.0.1"
WINDOW_SEC = 30
SAFE_BUFFER = 6000

def get_af_label_at(ann, start, end):
    mask = (ann.sample >= start) & (ann.sample < end)
    symbols = [ann.aux_note[i] for i in range(len(ann.sample)) if mask[i]]
    af_count = sum(1 for s in symbols if '(AFIB' in s or s.strip() == '(AFIB')
    return af_count > 0

def find_shdb_segments():
    hea_files = glob.glob(os.path.join(SHDB_PATH, "*.hea"))
    seg_af = None
    seg_nonaf = None
    meta_af = None
    meta_nonaf = None

    for hea in hea_files:
        if seg_af is not None and seg_nonaf is not None:
            break
        path = os.path.splitext(hea)[0]
        if not os.path.exists(path + '.atr'):
            continue
        try:
            record = wfdb.rdrecord(path)
            ann = wfdb.rdann(path, 'atr')
            fs = record.fs
            sig = record.p_signal
            n_samples = sig.shape[0]
            win = int(WINDOW_SEC * fs)

            rhythm_changes = [(ann.sample[i], ann.aux_note[i].strip()) for i in range(len(ann.sample)) if ann.aux_note[i].strip().startswith('(')]

            for k, (r_start, rhythm) in enumerate(rhythm_changes):
                if r_start < SAFE_BUFFER:
                    continue
                r_end = rhythm_changes[k + 1][0] if k + 1 < len(rhythm_changes) else n_samples
                if r_end - r_start < win:
                    continue

                chunk = sig[r_start:r_start + win, :]
                is_af = '(AFIB' in rhythm

                if is_af and seg_af is None:
                    seg_af = chunk
                    meta_af = (os.path.basename(path), record.sig_name, fs)
                elif not is_af and seg_nonaf is None:
                    seg_nonaf = chunk
                    meta_nonaf = (os.path.basename(path), record.sig_name, fs)

                if seg_af is not None and seg_nonaf is not None:
                    break

        except Exception:
            continue

    return seg_af, seg_nonaf, meta_af, meta_nonaf


def plot_shdb_raw():
    seg_af, seg_nonaf, meta_af, meta_nonaf = find_shdb_segments()

    if seg_af is None or seg_nonaf is None:
        print("Could not locate both AF and Non-AF segments in SHDB.")
        return

    fs_af = meta_af[2]
    fs_nonaf = meta_nonaf[2]
    time_af = np.arange(seg_af.shape[0]) / fs_af
    time_nonaf = np.arange(seg_nonaf.shape[0]) / fs_nonaf

    lead_labels = ["Modified CC5", "NASA"]

    fig, axes = plt.subplots(2, 2, figsize=(16, 8))
    fig.suptitle("Exploratory Data Analysis: SHDB-AF Raw ECG Segments", fontsize=14, weight='bold')

    row_labels = ["AF", "Non-AF"]
    segs = [seg_af, seg_nonaf]
    times = [time_af, time_nonaf]

    for row in range(2):
        for col in range(2):
            ax = axes[row, col]
            ax.plot(times[row], segs[row][:, col], color='black', linewidth=0.6)
            ax.set_xlim(0, WINDOW_SEC)
            ax.grid(alpha=0.3)
            if col == 0:
                ax.set_ylabel(row_labels[row], fontsize=11, weight='bold')
            if row == 0:
                ax.set_title(lead_labels[col], fontsize=11, weight='bold')
            if row == 1:
                ax.set_xlabel("Time (s)", fontsize=10)
            ax.set_ylabel(ax.get_ylabel() + ("\nAmplitude (mV)" if col == 0 else ""), fontsize=10)

    for row in range(2):
        axes[row, 0].set_ylabel(f"{row_labels[row]}\nAmplitude (mV)", fontsize=10, weight='bold')

    plt.tight_layout()
    plt.show()


plot_shdb_raw()

### IRIDIA-AF

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import h5py
import matplotlib.pyplot as plt
from scipy import signal as sp_signal

IRIDIA_PATH = "/content/drive/MyDrive/sp datasets/iridia-af-records-v1.0.1"
WINDOW_SEC = 30
NATIVE_FS = 200
SAFE_BUFFER = 10000


def find_iridia_segments():
    dirs = glob.glob(os.path.join(IRIDIA_PATH, "record_*"))
    seg_af = None
    seg_nonaf = None

    for d in dirs:
        if seg_af is not None and seg_nonaf is not None:
            break

        csvs = glob.glob(os.path.join(d, "*.csv"))
        target_csv = next((c for c in csvs if 'ecg' in os.path.basename(c)), csvs[0] if csvs else None)
        if not target_csv:
            continue

        try:
            df = pd.read_csv(target_csv)
            df.columns = df.columns.str.strip().str.replace('\ufeff', '')

            s_col = next((c for c in df.columns if 'start_qrs' in c), None)
            f_col = next((c for c in df.columns if 'file_index' in c), None)
            if not s_col or len(df) == 0:
                continue

            win = int(WINDOW_SEC * NATIVE_FS)

            # AF segment
            if seg_af is None:
                af_rows = df[df[s_col] > SAFE_BUFFER]
                if len(af_rows) == 0:
                    continue
                row = af_rows.iloc[0]
                af_start = int(row[s_col])
                af_fidx = int(row[f_col]) if f_col else 0
                h5_files = glob.glob(os.path.join(d, f"*_ecg_{af_fidx:02d}.h5"))
                if h5_files:
                    with h5py.File(h5_files[0], 'r') as f:
                        dset = f[list(f.keys())[0]]
                        if af_start + win <= dset.shape[0]:
                            chunk = dset[af_start:af_start + win]
                            if chunk.ndim == 1:
                                chunk = chunk[:, np.newaxis]
                            seg_af = chunk

            # Non-AF segment
            if seg_nonaf is None:
                first_af_file = df[f_col].min() if f_col else 0
                first_af_start = df[s_col].min()
                if first_af_file == 0 and first_af_start < 20000:
                    continue
                h5_norm = glob.glob(os.path.join(d, "*_ecg_00.h5"))
                if h5_norm:
                    with h5py.File(h5_norm[0], 'r') as f:
                        dset = f[list(f.keys())[0]]
                        r_start = SAFE_BUFFER
                        if r_start + win <= dset.shape[0]:
                            chunk = dset[r_start:r_start + win]
                            if chunk.ndim == 1:
                                chunk = chunk[:, np.newaxis]
                            seg_nonaf = chunk

        except Exception:
            continue

    return seg_af, seg_nonaf


def plot_iridia_raw():
    seg_af, seg_nonaf = find_iridia_segments()

    if seg_af is None or seg_nonaf is None:
        print("Could not locate both AF and Non-AF segments in IRIDIA.")
        return

    time = np.arange(int(WINDOW_SEC * NATIVE_FS)) / NATIVE_FS
    lead_labels = ["Lead I", "Lead II"]
    row_labels = ["AF", "Non-AF"]
    segs = [seg_af, seg_nonaf]

    fig, axes = plt.subplots(2, 2, figsize=(16, 8))
    fig.suptitle("Exploratory Data Analysis: IRIDIA-AF Raw ECG Segments", fontsize=14, weight='bold')

    for row in range(2):
        n_cols = segs[row].shape[1] if segs[row].ndim > 1 else 1
        for col in range(2):
            ax = axes[row, col]
            ch = col if col < n_cols else 0
            ax.plot(time, segs[row][:, ch], color='black', linewidth=0.6)
            ax.set_xlim(0, WINDOW_SEC)
            ax.grid(alpha=0.3)
            if row == 0:
                ax.set_title(lead_labels[col], fontsize=11, weight='bold')
            if row == 1:
                ax.set_xlabel("Time (s)", fontsize=10)

    for row in range(2):
        axes[row, 0].set_ylabel(f"{row_labels[row]}\nAmplitude (mV)", fontsize=10, weight='bold')

    plt.tight_layout()
    plt.show()


plot_iridia_raw()

# Preprocessing
Done through Google Colab/local

## Building the Dataset

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import wfdb
import h5py
import gc
from scipy import signal

# ==========================================
# 1. CONFIGURATION
# ==========================================
class Config:
    SHDB_PATH = "/content/drive/MyDrive/sp datasets/shdb-af-a-japanese-holter-ecg-database-of-atrial-fibrillation-1.0.1"
    IRIDIA_PATH = "/content/drive/MyDrive/sp datasets/iridia-af-records-v1.0.1"
    OUTPUT_DIR = "/content/drive/MyDrive/sp datasets/PROCESSED_TENSORS_V1"

    TARGET_FS = 125
    WINDOW_SAMPLES = 30 * 125 # 3750
    CHANNELS = 2

    AF_THRESHOLD = 0.5
    SAFE_BUFFER = 6000
    PAD_SEC = 5
    PAD_SAMPLES = PAD_SEC * 125

    CHUNK_SIZE = 20000

    LOW = 0.5 / (0.5 * 125); HIGH = 40.0 / (0.5 * 125)
    B, A = signal.butter(6, [LOW, HIGH], btype='band')

cfg = Config()

if not os.path.exists(cfg.OUTPUT_DIR): os.makedirs(cfg.OUTPUT_DIR)

# ==========================================
# 2. HELPER FUNCTIONS
# ==========================================
def process_segment(raw_padded_chunk):
    processed_leads = []
    for i in range(raw_padded_chunk.shape[1]):
        sig = raw_padded_chunk[:, i]
        sig = signal.filtfilt(cfg.B, cfg.A, sig)
        if len(sig) > 2 * cfg.PAD_SAMPLES:
            valid_sig = sig[cfg.PAD_SAMPLES : -cfg.PAD_SAMPLES]
        else: valid_sig = sig
        mu, sigma = np.mean(valid_sig), np.std(valid_sig)
        if sigma != 0: valid_sig = (valid_sig - mu) / sigma
        processed_leads.append(valid_sig)
    return np.stack(processed_leads, axis=-1)

def get_resume_state():
    existing = glob.glob(os.path.join(cfg.OUTPUT_DIR, "part_*.npz"))
    if not existing: return 0, 0
    existing.sort()
    last_file = existing[-1]
    last_idx = int(last_file.split('part_')[1].split('.npz')[0])
    items_done = (last_idx + 1) * cfg.CHUNK_SIZE

    print(f"RESUME: Found {len(existing)} parts (Last: {last_file})")
    print(f"   Skipping first {items_done:,} segments.")
    return last_idx + 1, items_done

def calculate_valid_windows(sig_len):
    """Calculates how many windows a file yields WITHOUT loading it."""
    count = 0
    total_wins = sig_len // cfg.WINDOW_SAMPLES
    for i in range(total_wins):
        s = i * cfg.WINDOW_SAMPLES
        e = s + cfg.WINDOW_SAMPLES
        # Exact logic match with generator
        if s < cfg.PAD_SAMPLES or e + cfg.PAD_SAMPLES > sig_len: continue
        if s < cfg.SAFE_BUFFER: continue
        count += 1
    return count

# ==========================================
# 3. GENERATORS (Smart File Skipping)
# ==========================================
def combined_generator(skip_count=0):
    global_yielded_count = 0

    # --- SHDB PHASE ---
    print(">>> Phase 1: SHDB-AF")
    files = glob.glob(os.path.join(cfg.SHDB_PATH, "*.hea"))
    for hea in files:
        pid = os.path.basename(hea).replace(".hea", "")
        group_id = f"SHDB_{pid}"
        if not os.path.exists(hea.replace('.hea', '.atr')): continue

        try:
            # 1. FAST CHECK: Can we skip this entire file?
            header = wfdb.rdheader(os.path.splitext(hea)[0])
            n_wins = calculate_valid_windows(header.sig_len)

            if global_yielded_count + n_wins <= skip_count:
                global_yielded_count += n_wins
                # print(f"   [Skipped SHDB_{pid}]") # Uncomment for verbose
                continue # SKIP FILE

            # 2. Process File (Only if needed)
            sig, _ = wfdb.rdsamp(os.path.splitext(hea)[0])
            ann = wfdb.rdann(os.path.splitext(hea)[0], 'atr')

            mask = np.zeros(len(sig), dtype=bool)
            state = False
            for i in range(len(ann.sample)-1):
                note = ann.aux_note[i]
                if '(AFIB' in note: state = True
                elif '(N' in note or '(AFL' in note or '(AT' in note: state = False
                if state: mask[ann.sample[i]:ann.sample[i+1]] = True
            if state: mask[ann.sample[-1]:] = True

            wins = len(sig) // cfg.WINDOW_SAMPLES
            for i in range(wins):
                s = i * cfg.WINDOW_SAMPLES; e = s + cfg.WINDOW_SAMPLES
                if s < cfg.PAD_SAMPLES or e + cfg.PAD_SAMPLES > len(sig): continue
                if s < cfg.SAFE_BUFFER: continue

                # Check resume counter
                if global_yielded_count < skip_count:
                    global_yielded_count += 1
                    continue

                burden = np.mean(mask[s:e])
                label = 1 if burden >= cfg.AF_THRESHOLD else 0

                raw = sig[s - cfg.PAD_SAMPLES : e + cfg.PAD_SAMPLES]
                yield process_segment(raw), label, group_id
                global_yielded_count += 1

        except Exception as e: continue

    # --- IRIDIA PHASE ---
    print("\n>>> Phase 2: IRIDIA-AF")
    dirs = glob.glob(os.path.join(cfg.IRIDIA_PATH, "record_*"))
    for d in dirs:
        rec_id = os.path.basename(d)
        group_id = f"IRIDIA_{rec_id}"

        try:
            # Setup Metadata
            csvs = glob.glob(os.path.join(d, "*.csv"))
            target_csv = next((c for c in csvs if 'ecg' in c), csvs[0] if csvs else None)
            if not target_csv: continue

            df = pd.read_csv(target_csv)
            df.columns = df.columns.str.strip().str.replace('\ufeff', '')

            s_col = next((c for c in df.columns if 'start_qrs' in c), None)
            if not s_col: s_col = next((c for c in df.columns if 'start' in c and 'index' in c), None)
            e_col = next((c for c in df.columns if 'end_qrs' in c), None)
            if not e_col: e_col = next((c for c in df.columns if 'end' in c and 'index' in c), None)
            f_col = next((c for c in df.columns if 'file_index' in c), None)
            if not s_col or not e_col: continue

            h5s = [f for f in glob.glob(os.path.join(d, "*.h5")) if "_rr_" not in f]
            for h in h5s:
                try: c_fidx = int(os.path.basename(h).split('_ecg_')[1].split('.')[0])
                except: c_fidx = 0

                # 1. FAST CHECK: Can we skip this H5?
                # We need the 125Hz length.
                # Length 125 = Length 200 * (125/200)
                with h5py.File(h, 'r') as f:
                    len_200 = f[list(f.keys())[0]].shape[0]

                len_125 = int(len_200 * cfg.TARGET_FS / 200)
                n_wins = calculate_valid_windows(len_125)

                if global_yielded_count + n_wins <= skip_count:
                    global_yielded_count += n_wins
                    # print(f"   [Skipped IRIDIA {rec_id} File {c_fidx}]") # Uncomment for verbose
                    continue # SKIP FILE

                # 2. Process File
                with h5py.File(h, 'r') as f:
                    # Optimize: Load as float32 to save RAM
                    sig_200 = f[list(f.keys())[0]][:].astype(np.float32)
                    if sig_200.ndim == 1: sig_200 = np.stack([sig_200, sig_200], axis=-1)

                    sig_125 = signal.resample(sig_200, len_125)
                    del sig_200 # Free RAM
                    gc.collect()

                    # Mask
                    mask = np.zeros(len_125, dtype=bool)
                    ratio = cfg.TARGET_FS / 200
                    sub_df = df[df[f_col] == c_fidx] if f_col else df

                    for _, r in sub_df.iterrows():
                        s_idx = max(0, int(r[s_col]*ratio))
                        e_idx = min(len_125, int(r[e_col]*ratio))
                        mask[s_idx:e_idx] = True

                    wins = len_125 // cfg.WINDOW_SAMPLES
                    for i in range(wins):
                        s = i * cfg.WINDOW_SAMPLES; e = s + cfg.WINDOW_SAMPLES
                        if s < cfg.PAD_SAMPLES or e + cfg.PAD_SAMPLES > len(sig_125): continue
                        if s < cfg.SAFE_BUFFER: continue

                        # Check resume counter
                        if global_yielded_count < skip_count:
                            global_yielded_count += 1
                            continue

                        burden = np.mean(mask[s:e])
                        label = 1 if burden >= cfg.AF_THRESHOLD else 0

                        raw = sig_125[s - cfg.PAD_SAMPLES : e + cfg.PAD_SAMPLES]
                        yield process_segment(raw), label, group_id
                        global_yielded_count += 1

                # Cleanup after file
                del sig_125
                del mask
                gc.collect()

        except Exception as e:
            # print(f"Err {d}: {e}")
            continue

# ==========================================
# 4. MAIN LOOP
# ==========================================
def main_chunked_builder():
    print("="*50)
    print("🚀 STARTING SMART RESUME BUILDER")
    print(f"   Output: {cfg.OUTPUT_DIR}")
    print("="*50)

    next_part_idx, skip_count = get_resume_state()

    buffer_X = []
    buffer_y = []
    buffer_groups = []

    for X, y, gid in combined_generator(skip_count=skip_count):
        buffer_X.append(X)
        buffer_y.append(y)
        buffer_groups.append(gid)

        if len(buffer_X) >= cfg.CHUNK_SIZE:
            filename = os.path.join(cfg.OUTPUT_DIR, f"part_{next_part_idx:03d}.npz")
            print(f"   💾 Saving {filename}...", end=" ")

            np.savez_compressed(filename,
                                X=np.array(buffer_X, dtype=np.float32),
                                y=np.array(buffer_y, dtype=np.int8),
                                groups=np.array(buffer_groups, dtype=str))

            print("Done.")
            next_part_idx += 1
            buffer_X, buffer_y, buffer_groups = [], [], []
            gc.collect()

    if len(buffer_X) > 0:
        filename = os.path.join(cfg.OUTPUT_DIR, f"part_{next_part_idx:03d}.npz")
        print(f"   💾 Saving Final Chunk {filename}...")
        np.savez_compressed(filename,
                            X=np.array(buffer_X, dtype=np.float32),
                            y=np.array(buffer_y, dtype=np.int8),
                            groups=np.array(buffer_groups, dtype=str))

    print("\nDATASET GENERATION COMPLETE.")

### Creating the HDF5 Master Dataset

In [ ]:
# CODE BLOCK 1: Consolidate to Master HDF5 (With Checkpointing)
import os
import json
import h5py
import numpy as np
from tqdm import tqdm
 
# Configuration
DATA_DIR = r"/content/drive/MyDrive/sp datasets"
OUTPUT_H5 = os.path.join(DATA_DIR, "afib_preprocessed.h5")
PROGRESS_FILE = os.path.join(DATA_DIR, "merge_progress.json")
 
def load_progress():
    """Load the list of already-processed files."""
    if os.path.exists(PROGRESS_FILE):
        with open(PROGRESS_FILE, 'r', encoding='utf-8') as f:
            return set(json.load(f))
    return set()
 
def save_progress(processed_files):
    """Save the list of processed files."""
    with open(PROGRESS_FILE, 'w', encoding='utf-8') as f:
        json.dump(list(processed_files), f, indent=2)
 
def consolidate_to_hdf5():
    print("="*60)
    print("CONSOLIDATING .npz FILES TO MASTER HDF5")
    print(f"   Output: {OUTPUT_H5}")
    print("="*60)
 
    # Find all part files
    import glob
    part_files = sorted(glob.glob(os.path.join(DATA_DIR, "part_*.npz")))
 
    if not part_files:
        print("No .npz files found!")
        return
 
    print(f"Found {len(part_files)} part files")
 
    # Load checkpoint
    processed_files = load_progress()
    print(f"Already processed: {len(processed_files)} files")
 
    # Initialize or open HDF5
    if os.path.exists(OUTPUT_H5) and len(processed_files) > 0:
        print("Resuming from existing HDF5...")
        mode = 'a'
    else:
        print("Creating new HDF5 file...")
        mode = 'w'
        processed_files = set()
 
    with h5py.File(OUTPUT_H5, mode) as h5f:
        # Create or get datasets
        if mode == 'w':
            # Create resizable datasets
            h5f.create_dataset('X', shape=(0, 3750, 2), maxshape=(None, 3750, 2), 
                             dtype='float32', chunks=(1000, 3750, 2))
            h5f.create_dataset('y', shape=(0,), maxshape=(None,), 
                             dtype='int8', chunks=(10000,))
            h5f.create_dataset('groups', shape=(0,), maxshape=(None,), 
                             dtype=h5py.string_dtype(encoding='utf-8'), chunks=(10000,))
            current_size = 0
        else:
            current_size = h5f['X'].shape[0]
            print(f"   Current dataset size: {current_size:,} samples")
 
        # Process each file
        for part_file in tqdm(part_files, desc="Merging"):
            filename = os.path.basename(part_file)
 
            # Skip if already processed
            if filename in processed_files:
                continue
 
            try:
                # Load data
                with np.load(part_file) as data:
                    X_batch = data['X']
                    y_batch = data['y']
                    groups_batch = data['groups']
 
                batch_size = len(X_batch)
 
                # Resize datasets
                new_size = current_size + batch_size
                h5f['X'].resize((new_size, 3750, 2))
                h5f['y'].resize((new_size,))
                h5f['groups'].resize((new_size,))
 
                # Append data
                h5f['X'][current_size:new_size] = X_batch
                h5f['y'][current_size:new_size] = y_batch
                h5f['groups'][current_size:new_size] = groups_batch.astype('S')
 
                # Flush to disk
                h5f.flush()
 
                # Update progress
                current_size = new_size
                processed_files.add(filename)
                save_progress(processed_files)
 
            except Exception as e:
                print(f"\nError processing {filename}: {e}")
                continue
 
        # Final report
        print("\n" + "="*60)
        print("CONSOLIDATION COMPLETE")
        print("="*60)
        print(f"Final Dataset Shapes:")
        print(f"  X (signals): {h5f['X'].shape}")
        print(f"  y (labels):  {h5f['y'].shape}")
        print(f"  groups (IDs): {h5f['groups'].shape}")
        print("="*60)
 
if __name__ == "__main__":
    consolidate_to_hdf5()

## Dataset Characteristics

### Dataset Summary
Done in Kaggle


In [ ]:
"""
30-Second Segment Counter
Counts windows per patient, per dataset prefix, and overall.
Also breaks down by label (Normal vs AFib).
"""

import h5py
import numpy as np
from collections import defaultdict

H5_FILE = "/kaggle/input/datasets/sheianne/shdbiridiaprocessed/afib_preprocessed.h5"

with h5py.File(H5_FILE, 'r') as f:
    groups = f['groups'][:]
    groups = [g.decode() if isinstance(g, bytes) else g for g in groups]
    labels = f['y'][:]
    T      = f['X'].shape[1]   # window length in samples

sr       = 200                 # Hz — change if your data differs
win_secs = T / sr
print(f"Window length: {T} samples @ {sr} Hz = {win_secs:.1f} seconds\n")

# ── Per-patient counts ────────────────────────────────────────────────────────
patient_normal  = defaultdict(int)
patient_afib    = defaultdict(int)

for g, lbl in zip(groups, labels):
    if int(lbl) == 1:
        patient_afib[g]   += 1
    else:
        patient_normal[g] += 1

all_patients = sorted(set(groups))

# ── Group by dataset prefix (e.g. "SHDB", "RBDB", "UVAF") ───────────────────
# Assumes patient IDs are formatted like "SHDB_014", "RBDB_005", etc.
dataset_windows = defaultdict(lambda: {'normal': 0, 'afib': 0, 'patients': 0})

for pid in all_patients:
    prefix = pid.split('_')[0]
    n = patient_normal[pid]
    a = patient_afib[pid]
    dataset_windows[prefix]['normal']   += n
    dataset_windows[prefix]['afib']     += a
    dataset_windows[prefix]['patients'] += 1

# ── Print per-patient table ───────────────────────────────────────────────────
print("=" * 72)
print(f"{'Patient':<16} {'Normal':>10} {'AFib':>10} {'Total':>10} {'AF%':>8}")
print("=" * 72)
for pid in all_patients:
    n = patient_normal[pid]
    a = patient_afib[pid]
    t = n + a
    print(f"{pid:<16} {n:>10,} {a:>10,} {t:>10,} {a/t*100:>7.1f}%")

# ── Print per-dataset summary ─────────────────────────────────────────────────
print("\n" + "=" * 72)
print(f"{'Dataset':<12} {'Patients':>10} {'Normal':>12} {'AFib':>12} "
      f"{'Total':>12} {'AF%':>8}")
print("=" * 72)

grand_n = grand_a = grand_p = 0
for prefix in sorted(dataset_windows):
    d  = dataset_windows[prefix]
    n, a, p = d['normal'], d['afib'], d['patients']
    t  = n + a
    grand_n += n; grand_a += a; grand_p += p
    print(f"{prefix:<12} {p:>10,} {n:>12,} {a:>12,} {t:>12,} {a/t*100:>7.1f}%")

print("-" * 72)
grand_t = grand_n + grand_a
print(f"{'TOTAL':<12} {grand_p:>10,} {grand_n:>12,} {grand_a:>12,} "
      f"{grand_t:>12,} {grand_a/grand_t*100:>7.1f}%")

print(f"\nTotal recording time: {grand_t * win_secs / 3600:.1f} hours")

### Class Imbalance Plot
Done in Kaggle

In [ ]:
import h5py
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

h5_path = "/kaggle/input/datasets/sheianne/shdbiridiaprocessed/afib_preprocessed.h5" 

# 1. Read the labels from the H5 file
with h5py.File(h5_path, 'r') as f:
    labels = f['y'][:] 

# 2. Count the occurrences of each class
non_af_count = np.sum(labels == 0)
af_count = np.sum(labels == 1)

print(f"Total Segments: {len(labels):,}")
print(f"Normal (Non-AF): {non_af_count:,}")
print(f"AFib: {af_count:,}")

# 3. Plotting the Bar Chart
classes = ['Normal (Non-AF)', 'Atrial Fibrillation (AFib)']
counts = [non_af_count, af_count]

plt.figure(figsize=(8, 6))

# Blue for Normal, Red for AFib
bars = plt.bar(classes, counts, color=['#2196F3', '#F44336'], edgecolor='black', alpha=0.8)

plt.title('Class Distribution: Normal vs. AFib Segments', fontsize=16, fontweight='bold', pad=20)
plt.ylabel('Number of Segments', fontsize=12, fontweight='bold')
plt.xlabel('Rhythm Class', fontsize=12, fontweight='bold')

plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)

# Fix y-axis to show full integers with comma formatting
formatter = FuncFormatter(lambda x, _: f'{int(x):,}')
plt.gca().yaxis.set_major_formatter(formatter)

# Add comma-formatted value labels on top of each bar
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width() / 2., 
             height + (max(counts) * 0.015),
             f'{int(height):,}',
             ha='center', va='bottom', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

### NumPy vs HDF5 File Shapes

In [ ]:
import numpy as np
import h5py

NPZ_PATH = "/content/drive/MyDrive/sp datasets/PROCESSED_TENSORS_V1/part_000.npz"
H5_PATH = "/content/drive/MyDrive/sp datasets/afib_preprocessed.h5"

print("="*60)
print("1. INVESTIGATING NPZ FILE (part_000.npz)")
print("="*60)

try:
    with np.load(NPZ_PATH) as data:
        print(f"Arrays stored in this NPZ: {data.files}\n")

        for key in data.files:
            array = data[key]
            print(f"--- Array: '{key}' ---")
            print(f"Shape: {array.shape}")
            print(f"Data type: {array.dtype}")
            print(f"Sample data (first 3 items):")

            # Print a clean snippet based on dimensions
            if array.ndim >= 2:
                print(f"{array[:3, ..., 0]}") # Show first channel for readability
            else:
                print(f"{array[:3]}")
            print()
except Exception as e:
    print(f"Error reading NPZ: {e}")

print("="*60)
print("2. INVESTIGATING HDF5 FILE (afib_preprocessed.h5)")
print("="*60)

try:
    with h5py.File(H5_PATH, 'r') as f:
        print(f"Datasets stored in this H5: {list(f.keys())}\n")

        for key in f.keys():
            dataset = f[key]
            print(f"--- Dataset: '{key}' ---")
            print(f"Shape: {dataset.shape}")
            print(f"Data type: {dataset.dtype}")
            print(f"Sample data (first 3 items):")

            if len(dataset.shape) >= 2:
                # Load only first 3 items into RAM to show
                sample = dataset[:3, ..., 0]
                print(f"{sample}")
            else:
                sample = dataset[:3]
                print(f"{sample}")
            print()
except Exception as e:
    print(f"Error reading H5: {e}")

## Visualization of Preprocessed ECG Segments

### SHDB-AF

In [ ]:
import os
import h5py
import numpy as np
import matplotlib.pyplot as plt

DATA_DIR = "/content/drive/MyDrive/sp datasets"
H5_FILE = os.path.join(DATA_DIR, "afib_preprocessed.h5")

FS = 125
N_SAMPLES = 3750
time = np.arange(N_SAMPLES) / FS


def plot_shdb_preprocessed():
    if not os.path.exists(H5_FILE):
        print("HDF5 file not found.")
        return

    with h5py.File(H5_FILE, 'r') as h5f:
        X = h5f['X']
        y = h5f['y'][:]
        groups = h5f['groups'][:].astype(str)

        idx_af = None
        idx_nonaf = None

        for i in range(len(groups)):
            if groups[i].startswith('SHDB_'):
                if idx_af is None and y[i] == 1:
                    idx_af = i
                if idx_nonaf is None and y[i] == 0:
                    idx_nonaf = i
            if idx_af is not None and idx_nonaf is not None:
                break

        if idx_af is None or idx_nonaf is None:
            print("Could not find both AF and Non-AF SHDB samples.")
            return

        sig_af = X[idx_af]
        sig_nonaf = X[idx_nonaf]

    lead_labels = ["Modified CC5", "NASA"]
    row_labels = ["AF", "Non-AF"]
    segs = [sig_af, sig_nonaf]

    fig, axes = plt.subplots(2, 2, figsize=(16, 8))
    fig.suptitle("Preprocessing: SHDB-AF ECG Segments", fontsize=14, weight='bold')

    for row in range(2):
        for col in range(2):
            ax = axes[row, col]
            ax.plot(time, segs[row][:, col], color='black', linewidth=0.6)
            ax.set_xlim(0, 30)
            ax.grid(alpha=0.3)
            if row == 0:
                ax.set_title(lead_labels[col], fontsize=11, weight='bold')
            if row == 1:
                ax.set_xlabel("Time (s)", fontsize=10)

    for row in range(2):
        axes[row, 0].set_ylabel(f"{row_labels[row]}\nNormalized Amplitude", fontsize=10, weight='bold')

    plt.tight_layout()
    plt.show()


plot_shdb_preprocessed()

### IRIDIA-AF

In [ ]:
import os
import h5py
import numpy as np
import matplotlib.pyplot as plt

DATA_DIR = "/content/drive/MyDrive/sp datasets"
H5_FILE = os.path.join(DATA_DIR, "afib_preprocessed.h5")

FS = 125
N_SAMPLES = 3750
time = np.arange(N_SAMPLES) / FS


def plot_iridia_preprocessed():
    if not os.path.exists(H5_FILE):
        print("HDF5 file not found.")
        return

    with h5py.File(H5_FILE, 'r') as h5f:
        X = h5f['X']
        y = h5f['y'][:]
        groups = h5f['groups'][:].astype(str)

        idx_af = None
        idx_nonaf = None

        for i in range(len(groups)):
            if groups[i].startswith('IRIDIA_'):
                if idx_af is None and y[i] == 1:
                    idx_af = i
                if idx_nonaf is None and y[i] == 0:
                    idx_nonaf = i
            if idx_af is not None and idx_nonaf is not None:
                break

        if idx_af is None or idx_nonaf is None:
            print("Could not find both AF and Non-AF IRIDIA samples.")
            return

        sig_af = X[idx_af]
        sig_nonaf = X[idx_nonaf]

    lead_labels = ["Lead I", "Lead II"]
    row_labels = ["AF", "Non-AF"]
    segs = [sig_af, sig_nonaf]

    fig, axes = plt.subplots(2, 2, figsize=(16, 8))
    fig.suptitle("Preprocessing: IRIDIA-AF ECG Segments", fontsize=14, weight='bold')

    for row in range(2):
        for col in range(2):
            ax = axes[row, col]
            ax.plot(time, segs[row][:, col], color='black', linewidth=0.6)
            ax.set_xlim(0, 30)
            ax.grid(alpha=0.3)
            if row == 0:
                ax.set_title(lead_labels[col], fontsize=11, weight='bold')
            if row == 1:
                ax.set_xlabel("Time (s)", fontsize=10)

    for row in range(2):
        axes[row, 0].set_ylabel(f"{row_labels[row]}\nNormalized Amplitude", fontsize=10, weight='bold')

    plt.tight_layout()
    plt.show()


plot_iridia_preprocessed()

## Train-test Splitting

In [ ]:
import os
import glob
import numpy as np
import json
import random
from collections import defaultdict
from tqdm import tqdm

# ==========================================
# 1. CONFIGURATION
# ==========================================
OUTPUT_DIR = "/content/drive/MyDrive/sp datasets/PROCESSED_TENSORS_V1"
SPLIT_FILE = os.path.join(OUTPUT_DIR, "dataset_split.json")
TEST_RATIO = 0.20  # 20% Test, 80% Train
SEED = 42          # Ensure reproducibility

random.seed(SEED)
np.random.seed(SEED)

def generate_split():
    print("="*50)
    print("🚀 GENERATING TRAIN/TEST SPLIT MAP")
    print(f"   Target: {int((1-TEST_RATIO)*100)}% Train / {int(TEST_RATIO*100)}% Test")
    print(f"   Mode:   Patient Independent & Source Stratified")
    print("="*50)

    # 1. SCAN DATASET INVENTORY
    # We need to know which patients exist and how much data they have.
    print("\n>>> Scanning .npz files for patient IDs...")
    files = sorted(glob.glob(os.path.join(OUTPUT_DIR, "part_*.npz")))

    if not files:
        print("No part_*.npz files found! Did the builder finish?")
        return

    patient_counts = defaultdict(int)
    patient_sources = {}

    for f in tqdm(files, desc="Inventorying"):
        try:
            # We only load 'groups' to be fast. We don't need X or y yet.
            with np.load(f) as data:
                groups = data['groups']

            # Count windows per patient
            unique, counts = np.unique(groups, return_counts=True)
            for grp, count in zip(unique, counts):
                grp_str = str(grp)
                patient_counts[grp_str] += count

                # Identify Source (SHDB or IRIDIA) based on string prefix
                if "SHDB" in grp_str:
                    patient_sources[grp_str] = "SHDB"
                elif "IRIDIA" in grp_str:
                    patient_sources[grp_str] = "IRIDIA"
                else:
                    patient_sources[grp_str] = "UNKNOWN"
        except Exception as e:
            print(f"Warning: Could not read {f}: {e}")

    # 2. SEPARATE BY SOURCE
    shdb_patients = [p for p, s in patient_sources.items() if s == "SHDB"]
    iridia_patients = [p for p, s in patient_sources.items() if s == "IRIDIA"]

    # Shuffle for randomness
    random.shuffle(shdb_patients)
    random.shuffle(iridia_patients)

    print(f"\nFound {len(patient_counts)} Unique Patients")
    print(f"   - SHDB:   {len(shdb_patients)}")
    print(f"   - IRIDIA: {len(iridia_patients)}")

    # 3. PERFORM SPLIT
    def split_list(lst, ratio):
        cut = int(len(lst) * ratio)
        return lst[cut:], lst[:cut] # Train, Test

    shdb_train, shdb_test = split_list(shdb_patients, TEST_RATIO)
    iridia_train, iridia_test = split_list(iridia_patients, TEST_RATIO)

    # Combine
    train_set = shdb_train + iridia_train
    test_set = shdb_test + iridia_test

    # 4. CALCULATE WINDOW STATS (Volume Balance)
    train_wins = sum(patient_counts[p] for p in train_set)
    test_wins = sum(patient_counts[p] for p in test_set)
    total_wins = train_wins + test_wins

    print("\n" + "="*50)
    print("SPLIT RESULTS")
    print("="*50)
    print(f"TRAIN SET: {len(train_set)} Patients")
    print(f"   - Windows: {train_wins:,} ({train_wins/total_wins*100:.1f}%)")
    print(f"   - SHDB:    {len(shdb_train)}")
    print(f"   - IRIDIA:  {len(iridia_train)}")
    print("-" * 30)
    print(f"TEST SET:  {len(test_set)} Patients")
    print(f"   - Windows: {test_wins:,} ({test_wins/total_wins*100:.1f}%)")
    print(f"   - SHDB:    {len(shdb_test)}")
    print(f"   - IRIDIA:  {len(iridia_test)}")
    print("="*50)

    # 5. SAVE SPLIT MAP
    split_data = {
        "train_patients": train_set,
        "test_patients": test_set,
        "metadata": {
            "total_windows": int(total_wins),
            "train_windows": int(train_wins),
            "test_windows": int(test_wins),
            "source_breakdown": {
                "SHDB_Total": len(shdb_patients),
                "IRIDIA_Total": len(iridia_patients)
            }
        }
    }

    with open(SPLIT_FILE, 'w') as f:
        json.dump(split_data, f, indent=4)

    print(f"\nSplit Map saved to: {SPLIT_FILE}")
    print("   (Load this JSON during training to filter patients)")

if __name__ == "__main__":
    generate_split()

# Full Training Code
Includes 3-fold CV and final test set evaluation

## Seeding

In [ ]:
def set_seed(seed=198):
    """Set seeds for reproducibility"""
    import random
    import numpy as np
    import torch
    
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  # multi-GPU
    
    # Make CuDNN deterministic (may reduce performance)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
    # Set Python hash seed
    os.environ['PYTHONHASHSEED'] = str(seed)

## DDNN (Cai et al.)

In [ ]:
import os
import json
import h5py
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, matthews_corrcoef,
                             confusion_matrix, precision_recall_curve, auc)
from sklearn.model_selection import KFold
from tqdm import tqdm
import time
from datetime import timedelta
import matplotlib.pyplot as plt
import warnings
import gc
import random
import threading
import shutil
warnings.filterwarnings('ignore')

SEED = 198

def seed_worker(worker_id):
    worker_seed = SEED + worker_id
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def setup_resume_environment(input_root, save_dir):
    """
    Copy checkpoint files from read-only input dataset to writable save directory.

    Args:
        input_root: Path to read-only input dataset directory
        save_dir: Path to writable save directory
    """
    print(f"\nSetting up resume environment...")
    print(f"   Input dataset path: {input_root}")
    print(f"   Save directory: {save_dir}")

    # Create save directory if it doesn't exist
    os.makedirs(save_dir, exist_ok=True)
    print(f"   Save directory created/verified")

    # Check if input dataset exists
    if not os.path.exists(input_root):
        print(f"   Input dataset not found - starting fresh training")
        return

    # Copy all .json and .pth files from input to save directory
    copied_files = 0
    for filename in os.listdir(input_root):
        if filename.endswith('.json') or filename.endswith('.pth'):
            src_path = os.path.join(input_root, filename)
            dst_path = os.path.join(save_dir, filename)

            # Only copy if destination doesn't exist or is older
            if not os.path.exists(dst_path):
                shutil.copy2(src_path, dst_path)
                print(f"   Copied: {filename}")
                copied_files += 1

    if copied_files == 0:
        print(f"   No checkpoint files found to copy - starting fresh training")
    else:
        print(f"   Successfully copied {copied_files} file(s)")


def plot_confusion_matrix(cm, title, save_path):
    """Plot and save confusion matrix"""
    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(cm, interpolation='nearest', cmap='Blues')
    ax.figure.colorbar(im, ax=ax, label='Count')

    classes = ['Normal', 'AFib']
    ax.set(xticks=np.arange(cm.shape[1]),
           yticks=np.arange(cm.shape[0]),
           xticklabels=classes, yticklabels=classes,
           title=title,
           ylabel='True Label',
           xlabel='Predicted Label')

    ax.set_title(title, fontsize=14, fontweight='bold', pad=20)
    ax.set_ylabel('True Label', fontsize=12)
    ax.set_xlabel('Predicted Label', fontsize=12)

    thresh = cm.max() / 2.
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, format(cm[i, j], 'd'),
                   ha="center", va="center",
                   color="white" if cm[i, j] > thresh else "black",
                   fontsize=14, fontweight='bold')

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"   Confusion matrix plot saved: {save_path}")


class SEBlock(nn.Module):
    """Squeeze-and-Excitation Block"""
    def __init__(self, channels, reduction=16):
        super(SEBlock, self).__init__()
        self.squeeze = nn.AdaptiveAvgPool1d(1)
        self.excitation = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _ = x.size()
        y = self.squeeze(x).view(b, c)
        y = self.excitation(y).view(b, c, 1)
        return x * y.expand_as(x)


class DenseBlock(nn.Module):
    """Dense Block with growth rate"""
    def __init__(self, in_channels, growth_rate, num_layers):
        super(DenseBlock, self).__init__()
        self.layers = nn.ModuleList()
        for i in range(num_layers):
            self.layers.append(
                nn.Sequential(
                    nn.BatchNorm1d(in_channels + i * growth_rate),
                    nn.ReLU(inplace=True),
                    nn.Conv1d(in_channels + i * growth_rate, growth_rate,
                             kernel_size=3, padding=1, bias=False)
                )
            )

    def forward(self, x):
        features = [x]
        for layer in self.layers:
            out = layer(torch.cat(features, 1))
            features.append(out)
        return torch.cat(features, 1)


class TransitionLayer(nn.Module):
    """Transition Layer with compression"""
    def __init__(self, in_channels, out_channels):
        super(TransitionLayer, self).__init__()
        self.bn = nn.BatchNorm1d(in_channels)
        self.conv = nn.Conv1d(in_channels, out_channels, kernel_size=1, bias=False)
        self.pool = nn.AvgPool1d(kernel_size=2, stride=2)

    def forward(self, x):
        x = self.conv(F.relu(self.bn(x)))
        x = self.pool(x)
        return x


class DDNN(nn.Module):
    """
    DDNN Architecture (Cai et al. 2020) - TRUE Lead-Agnostic Version
    - Accepts ANY single lead without knowing which one
    - Stem: Filter Concatenation (1x7, 1x15, 1x23)
    - Blocks: Dense Blocks with SE attention
    - Specs: Growth Rate=6, Reduction=0.5, Block Config=[2, 4, 6, 4]
    """
    def __init__(self, in_channels=1, growth_rate=6, block_config=[2, 4, 6, 4],
                 reduction=0.5, num_classes=1):
        super(DDNN, self).__init__()

        self.stem_conv1 = nn.Conv1d(in_channels, 16, kernel_size=7, padding=3, bias=False)
        self.stem_conv2 = nn.Conv1d(in_channels, 16, kernel_size=15, padding=7, bias=False)
        self.stem_conv3 = nn.Conv1d(in_channels, 16, kernel_size=23, padding=11, bias=False)
        self.stem_bn = nn.BatchNorm1d(48)
        self.stem_relu = nn.ReLU(inplace=True)
        self.stem_pool = nn.MaxPool1d(kernel_size=3, stride=2, padding=1)

        num_features = 48
        self.blocks = nn.ModuleList()
        self.se_blocks = nn.ModuleList()
        self.transitions = nn.ModuleList()

        for i, num_layers in enumerate(block_config):
            self.se_blocks.append(SEBlock(num_features, reduction=16))

            block = DenseBlock(num_features, growth_rate, num_layers)
            self.blocks.append(block)
            num_features += num_layers * growth_rate

            if i != len(block_config) - 1:
                out_features = int(num_features * reduction)
                trans = TransitionLayer(num_features, out_features)
                self.transitions.append(trans)
                num_features = out_features

        self.final_bn = nn.BatchNorm1d(num_features)
        self.global_pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(num_features, num_classes)

    def forward(self, x):
        x = x.permute(0, 2, 1)

        x1 = self.stem_conv1(x)
        x2 = self.stem_conv2(x)
        x3 = self.stem_conv3(x)
        x = torch.cat([x1, x2, x3], dim=1)
        x = self.stem_relu(self.stem_bn(x))
        x = self.stem_pool(x)

        for i, (se, block) in enumerate(zip(self.se_blocks, self.blocks)):
            x = se(x)
            x = block(x)
            if i < len(self.transitions):
                x = self.transitions[i](x)

        x = F.relu(self.final_bn(x))
        x = self.global_pool(x).squeeze(-1)
        x = self.fc(x)
        return x


class LeadAgnosticDataset(Dataset):
    """
    TRUE Lead-Agnostic Dataset
    - Randomly samples from BOTH leads during training
    - Model learns to detect AFib regardless of which lead
    - Each sample is a single lead (randomly chosen)
    """
    def __init__(self, h5_path, patient_ids, augment_with_both_leads=True):
        self.h5_path = h5_path
        self.patient_ids = set(patient_ids)
        self.augment_with_both_leads = augment_with_both_leads

        print(f"   Scanning HDF5 for {len(patient_ids)} patients (Lead-Agnostic Mode)...")
        with h5py.File(h5_path, 'r') as f:
            all_groups = f['groups'][:]
            all_groups = [g.decode('utf-8') if isinstance(g, bytes) else g
                         for g in all_groups]

            self.indices = [i for i, g in enumerate(all_groups)
                           if g in self.patient_ids]

        if self.augment_with_both_leads:
            print(f"   Dataset ready: {len(self.indices)} samples × 2 leads = {len(self.indices)*2} total samples")
        else:
            print(f"   Dataset ready: {len(self.indices)} samples (random lead per sample)")

    def __len__(self):
        if self.augment_with_both_leads:
            return len(self.indices) * 2
        return len(self.indices)

    def __getitem__(self, idx):
        if self.augment_with_both_leads:
            actual_idx = self.indices[idx % len(self.indices)]
            lead_idx = 0 if idx < len(self.indices) else 1
        else:
            actual_idx = self.indices[idx]
            lead_idx = np.random.randint(0, 2)

        with h5py.File(self.h5_path, 'r') as f:
            X = f['X'][actual_idx]
            y = f['y'][actual_idx]

        X = X[:, lead_idx:lead_idx+1]

        return torch.FloatTensor(X), torch.FloatTensor([y])


class RAMEfficientDataset(Dataset):
    """RAM-efficient dataset with auto-detection"""
    def __init__(self, h5_path, patient_ids, augment_with_both_leads=True,
                 preload_mode='auto', use_float16=True, max_ram_gb=20):
        self.h5_path = h5_path
        self.patient_ids = set(patient_ids)
        self.augment_with_both_leads = augment_with_both_leads
        self.use_float16 = use_float16
        self.max_ram_gb = max_ram_gb

        print(f"   Initializing RAM-Efficient Dataset...")
        print(f"      Mode: {preload_mode} | Float16: {use_float16} | Max RAM: {max_ram_gb} GB")

        with h5py.File(h5_path, 'r') as f:
            all_groups = f['groups'][:]
            all_groups = [g.decode('utf-8') if isinstance(g, bytes) else g
                         for g in all_groups]
            self.indices = [i for i, g in enumerate(all_groups)
                           if g in self.patient_ids]
            self.sample_shape = f['X'].shape[1:]

        print(f"      Found {len(self.indices)} matching samples")

        bytes_per_sample = np.prod(self.sample_shape) * (2 if use_float16 else 4)
        total_gb = (len(self.indices) * bytes_per_sample) / (1024**3)
        print(f"      Estimated RAM: {total_gb:.2f} GB")

        if preload_mode == 'auto':
            if total_gb < max_ram_gb * 0.7:
                preload_mode = 'full'
                print(f"      Auto-selected: FULL preload")
            elif total_gb < max_ram_gb * 1.2:
                preload_mode = 'chunk'
                print(f"      Auto-selected: CHUNK mode")
            else:
                preload_mode = 'lazy'
                print(f"      Auto-selected: LAZY mode")

        self.preload_mode = preload_mode
        self.X_cache = None
        self.y_cache = None
        self.chunk_cache = {}
        self.chunk_size = 10000

        if preload_mode == 'full':
            self._preload_full()

        if self.augment_with_both_leads:
            print(f"      Dataset ready: {len(self.indices)} samples × 2 leads = {len(self.indices)*2} total samples")
        else:
            print(f"      Dataset ready: {len(self.indices)} samples (random lead per sample)")

    def _preload_full(self):
        print(f"      Preloading into RAM...")
        dtype = np.float16 if self.use_float16 else np.float32

        self.X_cache = np.empty((len(self.indices), *self.sample_shape), dtype=dtype)
        self.y_cache = np.empty(len(self.indices), dtype=np.int8)

        batch_size = 50000
        with h5py.File(self.h5_path, 'r') as f:
            for i in range(0, len(self.indices), batch_size):
                end_idx = min(i + batch_size, len(self.indices))
                batch_indices = self.indices[i:end_idx]

                X_batch = f['X'][batch_indices]
                y_batch = f['y'][batch_indices]

                self.X_cache[i:end_idx] = X_batch.astype(dtype) if self.use_float16 else X_batch
                self.y_cache[i:end_idx] = y_batch

                if end_idx % 100000 == 0:
                    print(f"         {end_idx:,}/{len(self.indices):,}")

        print(f"      Preload complete!")
        gc.collect()

    def _load_chunk(self, chunk_id):
        if chunk_id in self.chunk_cache:
            return

        start_idx = chunk_id * self.chunk_size
        end_idx = min(start_idx + self.chunk_size, len(self.indices))
        chunk_indices = self.indices[start_idx:end_idx]

        dtype = np.float16 if self.use_float16 else np.float32

        with h5py.File(self.h5_path, 'r') as f:
            X_chunk = f['X'][chunk_indices]
            y_chunk = f['y'][chunk_indices]

            if self.use_float16:
                X_chunk = X_chunk.astype(np.float16)

            self.chunk_cache[chunk_id] = (X_chunk, y_chunk)

        if len(self.chunk_cache) > 3:
            oldest = min(self.chunk_cache.keys())
            del self.chunk_cache[oldest]
            gc.collect()

    def __len__(self):
        return len(self.indices) * 2 if self.augment_with_both_leads else len(self.indices)

    def __getitem__(self, idx):
        if self.augment_with_both_leads:
            actual_idx = idx % len(self.indices)
            lead_idx = 0 if idx < len(self.indices) else 1
        else:
            actual_idx = idx
            lead_idx = np.random.randint(0, 2)

        if self.preload_mode == 'full':
            X = self.X_cache[actual_idx]
            y = self.y_cache[actual_idx]
        elif self.preload_mode == 'chunk':
            chunk_id = actual_idx // self.chunk_size
            self._load_chunk(chunk_id)
            local_idx = actual_idx % self.chunk_size
            X, y_chunk = self.chunk_cache[chunk_id]
            X = X[local_idx]
            y = y_chunk[local_idx]
        else:
            with h5py.File(self.h5_path, 'r') as f:
                global_idx = self.indices[actual_idx]
                X = f['X'][global_idx]
                y = f['y'][global_idx]

        if self.use_float16 and isinstance(X, np.ndarray):
            X = X.astype(np.float32)

        X_lead = X[:, lead_idx:lead_idx+1]
        return torch.from_numpy(X_lead).float(), torch.tensor([y], dtype=torch.float32)


def test_dataloader_with_timeout(loader, timeout_seconds=30):
    """Test if dataloader works within timeout"""
    print(f"   Testing dataloader (timeout: {timeout_seconds}s)...")

    result = {'success': False, 'error': None}

    def load_batch():
        try:
            iterator = iter(loader)
            X, y = next(iterator)
            result['success'] = True
            result['batch_shape'] = X.shape
        except Exception as e:
            result['error'] = str(e)

    thread = threading.Thread(target=load_batch)
    thread.daemon = True
    thread.start()
    thread.join(timeout=timeout_seconds)

    if thread.is_alive():
        print(f"   DataLoader test TIMED OUT after {timeout_seconds}s")
        return False

    if result['success']:
        print(f"   DataLoader test PASSED (batch shape: {result['batch_shape']})")
        return True
    else:
        print(f"   DataLoader test FAILED: {result['error']}")
        return False


def create_adaptive_datasets(h5_path, train_patients, val_patients, batch_size=128):
    """Adaptive dataset creation with automatic fallback"""
    print("\n" + "="*80)
    print("ADAPTIVE DATASET CREATION")
    print("="*80)

    print("\nAttempting RAM-Efficient Mode...")
    try:
        train_dataset = RAMEfficientDataset(
            h5_path, train_patients,
            augment_with_both_leads=True,
            preload_mode='auto',
            use_float16=True,
            max_ram_gb=20
        )

        val_dataset = RAMEfficientDataset(
            h5_path, val_patients,
            augment_with_both_leads=True,
            preload_mode='auto',
            use_float16=True,
            max_ram_gb=20
        )

        g = torch.Generator()
        g.manual_seed(SEED)

        train_loader = DataLoader(
            train_dataset,
            batch_size=batch_size,
            shuffle=True,
            num_workers=2,
            worker_init_fn=seed_worker,
            generator=g,
            pin_memory=True,
            persistent_workers=True
        )

        if test_dataloader_with_timeout(train_loader, timeout_seconds=30):
            print("\nRAM-Efficient Mode SUCCESSFUL - Using with 2 workers")

            val_loader = DataLoader(
                val_dataset,
                batch_size=batch_size,
                shuffle=False,
                num_workers=2,
                pin_memory=True,
                persistent_workers=True
            )

            return train_loader, val_loader, 'ram_efficient'
        else:
            print("\nRAM-Efficient Mode got stuck - Falling back...")
            del train_loader, train_dataset, val_dataset
            gc.collect()

    except Exception as e:
        print(f"\nRAM-Efficient Mode failed: {e}")
        print("   Falling back to on-disk loading...")

    print("\nUsing On-Disk Loading Mode (Original)...")
    train_dataset = LeadAgnosticDataset(h5_path, train_patients, augment_with_both_leads=True)
    val_dataset = LeadAgnosticDataset(h5_path, val_patients, augment_with_both_leads=True)

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0
    )

    print("On-Disk Mode ready (num_workers=0)")
    return train_loader, val_loader, 'on_disk'


def calculate_metrics(y_true, y_pred_prob, threshold=0.5):
    """
    Calculate comprehensive metrics for binary classification.
    Main Metric: MCC (Matthews Correlation Coefficient)
    """
    y_pred = (y_pred_prob >= threshold).astype(int)

    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0

    mcc = matthews_corrcoef(y_true, y_pred)

    try:
        auc_roc = roc_auc_score(y_true, y_pred_prob)
    except:
        auc_roc = 0.0

    try:
        precision_curve, recall_curve, _ = precision_recall_curve(y_true, y_pred_prob)
        auprc = auc(recall_curve, precision_curve)
    except:
        auprc = 0.0

    return {
        'accuracy': acc,
        'precision': prec,
        'recall': rec,
        'sensitivity': rec,
        'specificity': spec,
        'f1': f1,
        'mcc': mcc,
        'auc_roc': auc_roc,
        'auprc': auprc,
        'confusion_matrix': cm
    }


def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    all_preds = []
    all_labels = []

    for X, y in tqdm(loader, desc="Training", leave=False):
        X, y = X.to(device), y.to(device)

        optimizer.zero_grad()
        outputs = model(X)
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds = torch.sigmoid(outputs).detach().cpu().tolist()
        labels = y.cpu().tolist()
        all_preds.extend(preds)
        all_labels.extend(labels)

    return total_loss / len(loader), np.array(all_preds).flatten(), np.array(all_labels).flatten()


def validate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for X, y in tqdm(loader, desc="Validating", leave=False):
            X, y = X.to(device), y.to(device)

            outputs = model(X)
            loss = criterion(outputs, y)

            total_loss += loss.item()
            preds = torch.sigmoid(outputs).cpu().tolist()
            labels = y.cpu().tolist()
            all_preds.extend(preds)
            all_labels.extend(labels)

    return total_loss / len(loader), np.array(all_preds).flatten(), np.array(all_labels).flatten()


def ddnn_3fold_cv():
    set_seed()

    print("="*80)
    print("DDNN 3-FOLD CROSS-VALIDATION")
    print("   (Lead-Agnostic Training)")
    print("="*80)

    DATA_DIR = "/kaggle/input/shdbiridiaprocessed"
    H5_FILE = os.path.join(DATA_DIR, "afib_preprocessed.h5")
    SPLIT_FILE = os.path.join(DATA_DIR, "dataset_split.json")
    SAVE_DIR = "/kaggle/working/models/ddnn_3fold"

    # Set up resume environment
    INPUT_DATASET_PATH = "/kaggle/input/datasets/sheianne/checkpoints"
    setup_resume_environment(INPUT_DATASET_PATH, SAVE_DIR)

    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Device: {DEVICE}")

    BATCH_SIZE = 128
    LEARNING_RATE = 1e-4
    NUM_EPOCHS = 50
    EARLY_STOP_PATIENCE = 15

    print(f"\nLoading split from: {SPLIT_FILE}")
    with open(SPLIT_FILE, 'r') as f:
        split = json.load(f)

    train_patients = np.array(split['train_patients'])
    test_patients = split['test_patients']

    print(f"   Train patients: {len(train_patients)}")
    print(f"   Test patients: {len(test_patients)}")

    print(f"\nCalculating class weights...")
    with h5py.File(H5_FILE, 'r') as f:
        all_labels = f['y'][:]
        total_samples = len(all_labels)

    pos_count = np.sum(all_labels)
    neg_count = len(all_labels) - pos_count
    pos_weight = neg_count / pos_count
    print(f"   Total samples: {total_samples:,}")
    print(f"   Positive weight: {pos_weight:.2f}")

    progress_file = os.path.join(SAVE_DIR, "ddnn_progress.json")

    if os.path.exists(progress_file):
        print(f"\nFound existing progress file: {progress_file}")
        with open(progress_file, 'r') as f:
            progress = json.load(f)
        print(f"   Resuming from Fold {progress['current_fold']}, Epoch {progress['current_epoch']}")
    else:
        progress = {
            'current_fold': 1,
            'current_epoch': 1,
            'completed_folds': [],
            'fold_results': []
        }
        print(f"\nNo existing progress found. Starting fresh.")

    kfold = KFold(n_splits=3, shuffle=True, random_state=42)
    fold_results = progress['fold_results']

    for fold, (train_idx, val_idx) in enumerate(kfold.split(train_patients), 1):
        if fold < progress['current_fold']:
            print(f"\nSkipping Fold {fold} (already completed)")
            continue

        if fold in progress['completed_folds']:
            print(f"\nSkipping Fold {fold} (already completed)")
            continue

        print("\n" + "="*80)
        print(f"FOLD {fold}/3")
        print("="*80)

        fold_train_patients = train_patients[train_idx].tolist()
        fold_val_patients = train_patients[val_idx].tolist()

        print(f"   Fold {fold} train patients: {len(fold_train_patients)}")
        print(f"   Fold {fold} val patients: {len(fold_val_patients)}")

        train_loader, val_loader, mode_used = create_adaptive_datasets(
            H5_FILE, fold_train_patients, fold_val_patients, BATCH_SIZE
        )

        print(f"\nUsing mode: {mode_used.upper()}")
        print(f"Train batches: {len(train_loader)}")
        print(f"Val batches: {len(val_loader)}")

        model = DDNN(in_channels=1, growth_rate=6, block_config=[2, 4, 6, 4],
                    reduction=0.5, num_classes=1).to(DEVICE)

        criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight]).to(DEVICE))
        optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)

        model_save_path = os.path.join(SAVE_DIR, f"ddnn_fold{fold}_best.pth")
        checkpoint_path = os.path.join(SAVE_DIR, f"ddnn_fold{fold}_checkpoint.pth")

        best_val_mcc = -1
        best_epoch = 0
        epochs_without_improvement = 0
        start_epoch = 1
        early_stopped = False

        if fold == progress['current_fold'] and os.path.exists(checkpoint_path):
            print(f"\nLoading checkpoint from: {checkpoint_path}")
            checkpoint = torch.load(checkpoint_path, weights_only=False)
            model.load_state_dict(checkpoint['model_state_dict'])
            optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
            start_epoch = checkpoint['epoch'] + 1
            best_val_mcc = checkpoint['best_mcc']
            best_epoch = checkpoint['best_epoch']
            epochs_without_improvement = checkpoint['epochs_without_improvement']
            early_stopped = checkpoint.get('early_stopped', False)
            print(f"   Resuming from epoch {start_epoch}")
            print(f"   Best MCC so far: {best_val_mcc:.4f} (epoch {best_epoch})")

            if early_stopped:                                       # <--- ADD THIS BLOCK
                print(f"   Early stopping was already triggered - skipping to fold evaluation")

        if not early_stopped:
            for epoch in range(start_epoch, NUM_EPOCHS + 1):
                print(f"\nFold {fold} - Epoch {epoch}/{NUM_EPOCHS}")

                train_loss, train_preds, train_labels = train_epoch(
                    model, train_loader, criterion, optimizer, DEVICE)

                val_loss, val_preds, val_labels = validate(
                    model, val_loader, criterion, DEVICE)

                train_metrics = calculate_metrics(train_labels, train_preds)
                val_metrics = calculate_metrics(val_labels, val_preds)

                if val_metrics['mcc'] > best_val_mcc:
                    best_val_mcc = val_metrics['mcc']
                    best_epoch = epoch
                    epochs_without_improvement = 0

                    torch.save({
                        'fold': fold,
                        'epoch': epoch,
                        'model_state_dict': model.state_dict(),
                        'optimizer_state_dict': optimizer.state_dict(),
                        'best_mcc': best_val_mcc,
                        'val_metrics': val_metrics,
                        'train_metrics': train_metrics,
                        'architecture': 'DDNN'
                    }, model_save_path)

                    improvement_msg = "New best MCC! Model saved."
                else:
                    epochs_without_improvement += 1
                    improvement_msg = f"No improvement ({epochs_without_improvement}/{EARLY_STOP_PATIENCE})"

                print(f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
                print(f"Train MCC: {train_metrics['mcc']:.4f} | Val MCC: {val_metrics['mcc']:.4f}")
                print(f"{improvement_msg}")

                # --- CORRECTED EARLY STOPPING LOGIC ---
                if epochs_without_improvement >= EARLY_STOP_PATIENCE:
                    early_stopped = True
                    print(f"\nEarly stopping triggered at epoch {epoch}")

                # Save checkpoint with the updated early_stopped flag
                torch.save({
                    'fold': fold,
                    'epoch': epoch,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'best_mcc': best_val_mcc,
                    'best_epoch': best_epoch,
                    'epochs_without_improvement': epochs_without_improvement,
                    'early_stopped': early_stopped,  # IMPORTANT: Save the True state
                    'val_metrics': val_metrics,
                    'train_metrics': train_metrics
                }, checkpoint_path)

                progress['current_fold'] = fold
                progress['current_epoch'] = epoch
                progress['early_stopped'] = early_stopped
                with open(progress_file, 'w') as f:
                    json.dump(progress, f, indent=2)

                if early_stopped:
                    break

        checkpoint = torch.load(model_save_path, weights_only=False)
        model.load_state_dict(checkpoint['model_state_dict'])

        _, val_preds, val_labels = validate(model, val_loader, criterion, DEVICE)
        final_metrics = calculate_metrics(val_labels, val_preds)

        print(f"\n" + "="*80)
        print(f"FOLD {fold} FINAL RESULTS")
        print("="*80)
        print(f"Best Epoch: {best_epoch}")

        print(f"\nConfusion Matrix:")
        cm = final_metrics['confusion_matrix']
        print(f"  TN: {cm[0,0]:6d}  |  FP: {cm[0,1]:6d}")
        print(f"  FN: {cm[1,0]:6d}  |  TP: {cm[1,1]:6d}")

        cm_plot_path = os.path.join(SAVE_DIR, f"confusion_matrix_fold{fold}.png")
        plot_confusion_matrix(cm, f"DDNN - Fold {fold} Confusion Matrix", cm_plot_path)

        print(f"\nClassification Metrics:")
        print(f"  Accuracy:    {final_metrics['accuracy']:.4f}")
        print(f"  Precision:   {final_metrics['precision']:.4f}")
        print(f"  Sensitivity: {final_metrics['sensitivity']:.4f}")
        print(f"  Specificity: {final_metrics['specificity']:.4f}")
        print(f"  F1-Score:    {final_metrics['f1']:.4f}")
        print(f"  MCC:         {final_metrics['mcc']:.4f}")
        print(f"  AUC-ROC:     {final_metrics['auc_roc']:.4f}")
        print(f"  AUPRC:       {final_metrics['auprc']:.4f}")

        # Convert confusion matrix to list for JSON serialization
        final_metrics_serializable = final_metrics.copy()
        final_metrics_serializable['confusion_matrix'] = final_metrics['confusion_matrix'].tolist()

        fold_results.append(final_metrics_serializable)

        progress['completed_folds'].append(fold)
        progress['fold_results'] = fold_results
        progress['current_fold'] = fold + 1
        progress['current_epoch'] = 1
        with open(progress_file, 'w') as f:
            json.dump(progress, f, indent=2)

        if os.path.exists(checkpoint_path):
            os.remove(checkpoint_path)

        print(f"\nFold {fold} completed and saved to progress.")

    print("\n" + "="*80)
    print("3-FOLD CROSS-VALIDATION SUMMARY")
    print("="*80)

    avg_cm = np.mean([r['confusion_matrix'] for r in fold_results], axis=0)
    print(f"\nAverage Confusion Matrix:")
    print(f"  TN: {avg_cm[0,0]:8.1f}  |  FP: {avg_cm[0,1]:8.1f}")
    print(f"  FN: {avg_cm[1,0]:8.1f}  |  TP: {avg_cm[1,1]:8.1f}")

    avg_cm_plot_path = os.path.join(SAVE_DIR, "confusion_matrix_average.png")
    plot_confusion_matrix(avg_cm.astype(int), "DDNN - Average Confusion Matrix (3-Fold CV)", avg_cm_plot_path)

    print(f"\nAverage Classification Metrics (across 3 folds):")
    for metric in ['accuracy', 'precision', 'sensitivity', 'specificity', 'f1', 'mcc', 'auc_roc', 'auprc']:
        values = [r[metric] for r in fold_results]
        mean_val = np.mean(values)
        std_val = np.std(values)
        print(f"  {metric.capitalize():12s}: {mean_val:.4f} ± {std_val:.4f}")

    print("\n" + "="*80)
    print(f"Models saved in: {SAVE_DIR}")
    print("="*80)

# --- FINAL TRAINING START ---
    print("\n" + "="*40 + "\nFINAL TRAINING (Robust Resume)\n" + "="*40)

    # 1. Setup Datasets (Train + Test)
    final_train_loader, final_test_loader, _ = create_adaptive_datasets(
        H5_FILE, train_patients, test_patients, BATCH_SIZE
    )

    # 2. Reset Model (DDNN) & Optimizer
    final_model = DDNN(in_channels=1, growth_rate=6, block_config=[2, 4, 6, 4],
                       reduction=0.5, num_classes=1).to(DEVICE)

    optimizer = torch.optim.Adam(final_model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight]).to(DEVICE))

    final_save_path = os.path.join(SAVE_DIR, "ddnn_FINAL.pth")
    progress_file = os.path.join(SAVE_DIR, "ddnn_progress.json")

    best_mcc = -1
    no_improve = 0
    start_epoch = 1

    # --- RESUME LOGIC ---
    # Check if a final model already exists to resume from
    if os.path.exists(final_save_path):
        print(f"\nFound existing final checkpoint: {final_save_path}")
        try:
            checkpoint = torch.load(final_save_path, weights_only=False)
            final_model.load_state_dict(checkpoint['model'])

            # Recover epoch and MCC
            if 'epoch' in checkpoint:
                start_epoch = checkpoint['epoch'] + 1

            if 'metrics' in checkpoint and 'mcc' in checkpoint['metrics']:
                best_mcc = checkpoint['metrics']['mcc']

            print(f"   Resuming from Epoch {start_epoch}")
            print(f"   Current Best MCC: {best_mcc:.4f}")

            # --- JSON PATCHING ---
            # If we are resuming, ensure the JSON matches the checkpoint reality
            if os.path.exists(progress_file):
                with open(progress_file, 'r') as f:
                    progress = json.load(f)

                # If JSON says Epoch 1 but we are at Epoch 34, fix the JSON
                if progress.get('current_epoch', 0) < start_epoch - 1:
                    print(f"   [FIX] Patching stale JSON: {progress.get('current_epoch')} -> {start_epoch - 1}")
                    progress['current_epoch'] = start_epoch - 1
                    progress['early_stopped'] = False
                    with open(progress_file, 'w') as f:
                        json.dump(progress, f, indent=2)

        except Exception as e:
            print(f"   Error loading checkpoint: {e}")
            print(f"   Starting fresh from Epoch 1")
    else:
        # Initialize JSON if starting fresh
        if os.path.exists(progress_file):
            with open(progress_file, 'r') as f:
                progress = json.load(f)
        else:
            progress = {
                "current_fold": 4,
                "current_epoch": 0,
                "completed_folds": [1, 2, 3],
                "fold_results": [],
                "early_stopped": False
            }

    # 3. Train Loop
    for epoch in range(start_epoch, NUM_EPOCHS + 1):
        print(f"\nFinal Epoch {epoch}/{NUM_EPOCHS}")

        train_loss, train_p, train_l = train_epoch(final_model, final_train_loader, criterion, optimizer, DEVICE)
        test_loss, test_p, test_l = validate(final_model, final_test_loader, criterion, DEVICE)

        train_mcc = matthews_corrcoef(train_l, (train_p > 0.5).astype(int))
        test_metrics = calculate_metrics(test_l, test_p)

        print(f"Loss: {train_loss:.4f} (Train) | {test_loss:.4f} (Test)")
        print(f"MCC:  {train_mcc:.4f} (Train) | {test_metrics['mcc']:.4f} (Test)")

        # --- UPDATE JSON LIVE ---
        progress['current_epoch'] = epoch
        with open(progress_file, 'w') as f:
            json.dump(progress, f, indent=2)
        # ------------------------

        if test_metrics['mcc'] > best_mcc:
            best_mcc = test_metrics['mcc']
            no_improve = 0
            torch.save({
                'epoch': epoch,
                'model': final_model.state_dict(),
                'metrics': test_metrics,
                'architecture': 'DDNN'
            }, final_save_path)
            print(">> Saved New Best Model")
        else:
            no_improve += 1
            print(f">> No improve ({no_improve}/{EARLY_STOP_PATIENCE})")

        if no_improve >= EARLY_STOP_PATIENCE:
            print("Stopping Early.")
            progress['early_stopped'] = True
            with open(progress_file, 'w') as f:
                json.dump(progress, f, indent=2)
            break

    # 4. Final Evaluation
    print("\n" + "="*40 + "\nFINAL TEST REPORT\n" + "="*40)

    # Load best model safely (weights_only=False to prevent UnpicklingError)
    if os.path.exists(final_save_path):
        checkpoint = torch.load(final_save_path, weights_only=False)
        final_model.load_state_dict(checkpoint['model'])

    _, final_p, final_l = validate(final_model, final_test_loader, criterion, DEVICE)
    final_metrics = calculate_metrics(final_l, final_p)

    plot_confusion_matrix(final_metrics['confusion_matrix'], "DDNN - Final Test Matrix", os.path.join(SAVE_DIR, "cm_final_ddnn.png"))

    print(f"\nClassification Metrics:")
    print(f"  Accuracy:    {final_metrics['accuracy']:.4f}")
    print(f"  Precision:   {final_metrics['precision']:.4f}")
    print(f"  Sensitivity: {final_metrics['sensitivity']:.4f}")
    print(f"  Specificity: {final_metrics['specificity']:.4f}")
    print(f"  F1-Score:    {final_metrics['f1']:.4f}")
    print(f"  MCC:         {final_metrics['mcc']:.4f}")
    print(f"  AUC-ROC:     {final_metrics['auc_roc']:.4f}")
    print(f"  AUPRC:       {final_metrics['auprc']:.4f}")

    # Save final metrics
    with open(os.path.join(SAVE_DIR, "final_test_results.json"), 'w') as f:
        serializable_mets = {k: (v.tolist() if isinstance(v, np.ndarray) else v) for k, v in final_metrics.items()}
        json.dump(serializable_mets, f, indent=2)

if __name__ == "__main__":
    ddnn_3fold_cv()

## RawECGNet (Ben-Moshe et al.)

In [ ]:
import os
import json
import h5py
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, roc_auc_score, matthews_corrcoef,
                             confusion_matrix, precision_recall_curve, auc)
from sklearn.model_selection import KFold
from tqdm import tqdm
import time
from datetime import timedelta
import matplotlib.pyplot as plt
import warnings
import gc
import random
import threading
import shutil
warnings.filterwarnings('ignore')

SEED = 198

def seed_worker(worker_id):
    worker_seed = SEED + worker_id
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def setup_resume_environment(input_root, save_dir):
    """
    Copy checkpoint files from read-only input dataset to writable save directory.
    
    Args:
        input_root: Path to read-only input dataset directory
        save_dir: Path to writable save directory
    """
    print(f"\nSetting up resume environment...")
    print(f"   Input dataset path: {input_root}")
    print(f"   Save directory: {save_dir}")
    
    # Create save directory if it doesn't exist
    os.makedirs(save_dir, exist_ok=True)
    print(f"   Save directory created/verified")
    
    # Check if input dataset exists
    if not os.path.exists(input_root):
        print(f"   Input dataset not found - starting fresh training")
        return
    
    # Copy all .json and .pth files from input to save directory
    copied_files = 0
    for filename in os.listdir(input_root):
        if filename.endswith('.json') or filename.endswith('.pth'):
            src_path = os.path.join(input_root, filename)
            dst_path = os.path.join(save_dir, filename)
            
            # Only copy if destination doesn't exist or is older
            if not os.path.exists(dst_path):
                shutil.copy2(src_path, dst_path)
                print(f"   Copied: {filename}")
                copied_files += 1
    
    if copied_files == 0:
        print(f"   No checkpoint files found to copy - starting fresh training")
    else:
        print(f"   Successfully copied {copied_files} file(s)")


def plot_confusion_matrix(cm, title, save_path):
    """Plot and save confusion matrix"""
    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(cm, interpolation='nearest', cmap='Blues')
    ax.figure.colorbar(im, ax=ax, label='Count')
    
    classes = ['Normal', 'AFib']
    ax.set(xticks=np.arange(cm.shape[1]),
           yticks=np.arange(cm.shape[0]),
           xticklabels=classes, yticklabels=classes,
           title=title,
           ylabel='True Label',
           xlabel='Predicted Label')
    
    ax.set_title(title, fontsize=14, fontweight='bold', pad=20)
    ax.set_ylabel('True Label', fontsize=12)
    ax.set_xlabel('Predicted Label', fontsize=12)
    
    thresh = cm.max() / 2.
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, format(cm[i, j], 'd'),
                   ha="center", va="center",
                   color="white" if cm[i, j] > thresh else "black",
                   fontsize=14, fontweight='bold')
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"   Confusion matrix plot saved: {save_path}")


class DSU(nn.Module):
    """
    Domain-Shifts-with-Uncertainty Layer
    Adds statistical noise during training to improve generalization
    """
    def __init__(self, p=0.5):
        super(DSU, self).__init__()
        self.p = p
    
    def forward(self, x):
        if not self.training or np.random.rand() > self.p:
            return x
        
        batch_size = x.size(0)
        mean = x.mean(dim=[2], keepdim=True)
        std = x.std(dim=[2], keepdim=True) + 1e-8
        
        noise = torch.randn_like(x) * std + mean
        
        return noise


class ResBlock(nn.Module):
    """Standard 1D ResNet Block"""
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, dropout=0.2):
        super(ResBlock, self).__init__()
        
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size, 
                              stride=stride, padding=kernel_size//2, bias=False)
        self.bn1 = nn.BatchNorm1d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.dropout = nn.Dropout(dropout)
        
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size, 
                              stride=1, padding=kernel_size//2, bias=False)
        self.bn2 = nn.BatchNorm1d(out_channels)
        
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv1d(in_channels, out_channels, 1, stride=stride, bias=False),
                nn.BatchNorm1d(out_channels)
            )
        else:
            self.shortcut = nn.Identity()
    
    def forward(self, x):
        identity = self.shortcut(x)
        
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.dropout(out)
        
        out = self.conv2(out)
        out = self.bn2(out)
        
        out += identity
        out = self.relu(out)
        
        return out


class ShrinkBlock(nn.Module):
    """Reduces channel dimensions"""
    def __init__(self, in_channels, out_channels):
        super(ShrinkBlock, self).__init__()
        self.conv = nn.Conv1d(in_channels, out_channels, 1, bias=False)
        self.bn = nn.BatchNorm1d(out_channels)
        self.relu = nn.ReLU(inplace=True)
    
    def forward(self, x):
        return self.relu(self.bn(self.conv(x)))


class RawECGNet(nn.Module):
    """
    RawECGNet Architecture (Ben-Moshe et al. 2024) - Lead-Agnostic Version
    - Accepts ANY single lead without knowing which one
    - Uses DSU for domain shift robustness
    - ResNet-based encoder with BiGRU temporal head
    """
    def __init__(self, in_channels=1, num_classes=1, dropout=0.2):
        super(RawECGNet, self).__init__()
        
        self.stem = nn.Sequential(
            nn.Conv1d(in_channels, 16, kernel_size=7, stride=2, padding=3, bias=False),
            nn.BatchNorm1d(16),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(kernel_size=3, stride=2, padding=1)
        )
        
        self.encoder = nn.ModuleList()
        channels = [16, 32, 32, 64, 64, 128, 128]
        for i in range(7):
            in_ch = 16 if i == 0 else channels[i-1]
            out_ch = channels[i]
            stride = 2 if i in [1, 3, 5] else 1
            
            self.encoder.append(nn.Sequential(
                ResBlock(in_ch, out_ch, stride=stride, dropout=dropout),
                DSU(p=0.5)
            ))
        
        self.shrink = nn.ModuleList([
            ShrinkBlock(128, 96),
            ShrinkBlock(96, 64),
            ShrinkBlock(64, 48),
            ShrinkBlock(48, 32)
        ])
        
        self.feature_size = 117 * 32
        
        self.dense = nn.Sequential(
            nn.Linear(self.feature_size, 256),
            nn.BatchNorm1d(256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout(dropout),
            
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout(dropout),
            
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout(dropout)
        )
        
        self.gru = nn.GRU(32, 32, num_layers=2, batch_first=True, 
                         bidirectional=True, dropout=dropout if dropout > 0 else 0)
        
        self.classifier = nn.Sequential(
            nn.BatchNorm1d(64),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout(dropout),
            nn.Linear(64, num_classes)
        )
    
    def forward(self, x):
        x = x.permute(0, 2, 1)
        
        x = self.stem(x)
        
        for block in self.encoder:
            x = block(x)
        
        for shrink in self.shrink:
            x = shrink(x)
        
        x = x.permute(0, 2, 1)
        x, _ = self.gru(x)
        x = x[:, -1, :]
        
        x = self.classifier(x)
        
        return x


class LeadAgnosticDataset(Dataset):
    """
    TRUE Lead-Agnostic Dataset
    - Randomly samples from BOTH leads during training
    - Model learns to detect AFib regardless of which lead
    - Each sample is a single lead (randomly chosen)
    """
    def __init__(self, h5_path, patient_ids, augment_with_both_leads=True):
        self.h5_path = h5_path
        self.patient_ids = set(patient_ids)
        self.augment_with_both_leads = augment_with_both_leads
        
        print(f"   Scanning HDF5 for {len(patient_ids)} patients (Lead-Agnostic Mode)...")
        with h5py.File(h5_path, 'r') as f:
            all_groups = f['groups'][:]
            all_groups = [g.decode('utf-8') if isinstance(g, bytes) else g 
                         for g in all_groups]
            
            self.indices = [i for i, g in enumerate(all_groups) 
                           if g in self.patient_ids]
        
        if self.augment_with_both_leads:
            print(f"   Dataset ready: {len(self.indices)} samples × 2 leads = {len(self.indices)*2} total samples")
        else:
            print(f"   Dataset ready: {len(self.indices)} samples (random lead per sample)")
    
    def __len__(self):
        if self.augment_with_both_leads:
            return len(self.indices) * 2
        return len(self.indices)
    
    def __getitem__(self, idx):
        if self.augment_with_both_leads:
            actual_idx = self.indices[idx % len(self.indices)]
            lead_idx = 0 if idx < len(self.indices) else 1
        else:
            actual_idx = self.indices[idx]
            lead_idx = np.random.randint(0, 2)
        
        with h5py.File(self.h5_path, 'r') as f:
            X = f['X'][actual_idx]
            y = f['y'][actual_idx]
        
        X = X[:, lead_idx:lead_idx+1]
        
        return torch.FloatTensor(X), torch.FloatTensor([y])


class RAMEfficientDataset(Dataset):
    """RAM-efficient dataset with auto-detection"""
    def __init__(self, h5_path, patient_ids, augment_with_both_leads=True, 
                 preload_mode='auto', use_float16=True, max_ram_gb=20):
        self.h5_path = h5_path
        self.patient_ids = set(patient_ids)
        self.augment_with_both_leads = augment_with_both_leads
        self.use_float16 = use_float16
        self.max_ram_gb = max_ram_gb
        
        print(f"   Initializing RAM-Efficient Dataset...")
        print(f"      Mode: {preload_mode} | Float16: {use_float16} | Max RAM: {max_ram_gb} GB")
        
        with h5py.File(h5_path, 'r') as f:
            all_groups = f['groups'][:]
            all_groups = [g.decode('utf-8') if isinstance(g, bytes) else g 
                         for g in all_groups]
            self.indices = [i for i, g in enumerate(all_groups) 
                           if g in self.patient_ids]
            self.sample_shape = f['X'].shape[1:]
        
        print(f"      Found {len(self.indices)} matching samples")
        
        bytes_per_sample = np.prod(self.sample_shape) * (2 if use_float16 else 4)
        total_gb = (len(self.indices) * bytes_per_sample) / (1024**3)
        print(f"      Estimated RAM: {total_gb:.2f} GB")
        
        if preload_mode == 'auto':
            if total_gb < max_ram_gb * 0.7:
                preload_mode = 'full'
                print(f"      Auto-selected: FULL preload")
            elif total_gb < max_ram_gb * 1.2:
                preload_mode = 'chunk'
                print(f"      Auto-selected: CHUNK mode")
            else:
                preload_mode = 'lazy'
                print(f"      Auto-selected: LAZY mode")
        
        self.preload_mode = preload_mode
        self.X_cache = None
        self.y_cache = None
        self.chunk_cache = {}
        self.chunk_size = 10000
        
        if preload_mode == 'full':
            self._preload_full()
        
        if self.augment_with_both_leads:
            print(f"      Dataset ready: {len(self.indices)} samples × 2 leads = {len(self.indices)*2} total samples")
        else:
            print(f"      Dataset ready: {len(self.indices)} samples (random lead per sample)")
    
    def _preload_full(self):
        print(f"      Preloading into RAM...")
        dtype = np.float16 if self.use_float16 else np.float32
        
        self.X_cache = np.empty((len(self.indices), *self.sample_shape), dtype=dtype)
        self.y_cache = np.empty(len(self.indices), dtype=np.int8)
        
        batch_size = 50000
        with h5py.File(self.h5_path, 'r') as f:
            for i in range(0, len(self.indices), batch_size):
                end_idx = min(i + batch_size, len(self.indices))
                batch_indices = self.indices[i:end_idx]
                
                X_batch = f['X'][batch_indices]
                y_batch = f['y'][batch_indices]
                
                self.X_cache[i:end_idx] = X_batch.astype(dtype) if self.use_float16 else X_batch
                self.y_cache[i:end_idx] = y_batch
                
                if end_idx % 100000 == 0:
                    print(f"         {end_idx:,}/{len(self.indices):,}")
        
        print(f"      Preload complete!")
        gc.collect()
    
    def _load_chunk(self, chunk_id):
        if chunk_id in self.chunk_cache:
            return
        
        start_idx = chunk_id * self.chunk_size
        end_idx = min(start_idx + self.chunk_size, len(self.indices))
        chunk_indices = self.indices[start_idx:end_idx]
        
        dtype = np.float16 if self.use_float16 else np.float32
        
        with h5py.File(self.h5_path, 'r') as f:
            X_chunk = f['X'][chunk_indices]
            y_chunk = f['y'][chunk_indices]
            
            if self.use_float16:
                X_chunk = X_chunk.astype(np.float16)
            
            self.chunk_cache[chunk_id] = (X_chunk, y_chunk)
        
        if len(self.chunk_cache) > 3:
            oldest = min(self.chunk_cache.keys())
            del self.chunk_cache[oldest]
            gc.collect()
    
    def __len__(self):
        return len(self.indices) * 2 if self.augment_with_both_leads else len(self.indices)
    
    def __getitem__(self, idx):
        if self.augment_with_both_leads:
            actual_idx = idx % len(self.indices)
            lead_idx = 0 if idx < len(self.indices) else 1
        else:
            actual_idx = idx
            lead_idx = np.random.randint(0, 2)
        
        if self.preload_mode == 'full':
            X = self.X_cache[actual_idx]
            y = self.y_cache[actual_idx]
        elif self.preload_mode == 'chunk':
            chunk_id = actual_idx // self.chunk_size
            self._load_chunk(chunk_id)
            local_idx = actual_idx % self.chunk_size
            X, y_chunk = self.chunk_cache[chunk_id]
            X = X[local_idx]
            y = y_chunk[local_idx]
        else:
            with h5py.File(self.h5_path, 'r') as f:
                global_idx = self.indices[actual_idx]
                X = f['X'][global_idx]
                y = f['y'][global_idx]
        
        if self.use_float16 and isinstance(X, np.ndarray):
            X = X.astype(np.float32)
        
        X_lead = X[:, lead_idx:lead_idx+1]
        return torch.from_numpy(X_lead).float(), torch.tensor([y], dtype=torch.float32)


def test_dataloader_with_timeout(loader, timeout_seconds=30):
    """Test if dataloader works within timeout"""
    print(f"   Testing dataloader (timeout: {timeout_seconds}s)...")
    
    result = {'success': False, 'error': None}
    
    def load_batch():
        try:
            iterator = iter(loader)
            X, y = next(iterator)
            result['success'] = True
            result['batch_shape'] = X.shape
        except Exception as e:
            result['error'] = str(e)
    
    thread = threading.Thread(target=load_batch)
    thread.daemon = True
    thread.start()
    thread.join(timeout=timeout_seconds)
    
    if thread.is_alive():
        print(f"   DataLoader test TIMED OUT after {timeout_seconds}s")
        return False
    
    if result['success']:
        print(f"   DataLoader test PASSED (batch shape: {result['batch_shape']})")
        return True
    else:
        print(f"   DataLoader test FAILED: {result['error']}")
        return False


def create_adaptive_datasets(h5_path, train_patients, val_patients, batch_size=128):
    """Adaptive dataset creation with automatic fallback"""
    print("\n" + "="*80)
    print("ADAPTIVE DATASET CREATION")
    print("="*80)
    
    print("\nAttempting RAM-Efficient Mode...")
    try:
        train_dataset = RAMEfficientDataset(
            h5_path, train_patients,
            augment_with_both_leads=True,
            preload_mode='auto',
            use_float16=True,
            max_ram_gb=20
        )
        
        val_dataset = RAMEfficientDataset(
            h5_path, val_patients,
            augment_with_both_leads=True,
            preload_mode='auto',
            use_float16=True,
            max_ram_gb=20
        )
        
        g = torch.Generator()
        g.manual_seed(SEED)
        
        train_loader = DataLoader(
            train_dataset,
            batch_size=batch_size,
            shuffle=True,
            num_workers=2,
            worker_init_fn=seed_worker,
            generator=g,
            pin_memory=True,
            persistent_workers=True
        )
        
        if test_dataloader_with_timeout(train_loader, timeout_seconds=30):
            print("\nRAM-Efficient Mode SUCCESSFUL - Using with 2 workers")
            
            val_loader = DataLoader(
                val_dataset,
                batch_size=batch_size,
                shuffle=False,
                num_workers=2,
                pin_memory=True,
                persistent_workers=True
            )
            
            return train_loader, val_loader, 'ram_efficient'
        else:
            print("\nRAM-Efficient Mode got stuck - Falling back...")
            del train_loader, train_dataset, val_dataset
            gc.collect()
    
    except Exception as e:
        print(f"\nRAM-Efficient Mode failed: {e}")
        print("   Falling back to on-disk loading...")
    
    print("\nUsing On-Disk Loading Mode (Original)...")
    train_dataset = LeadAgnosticDataset(h5_path, train_patients, augment_with_both_leads=True)
    val_dataset = LeadAgnosticDataset(h5_path, val_patients, augment_with_both_leads=True)
    
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0
    )
    
    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0
    )
    
    print("On-Disk Mode ready (num_workers=0)")
    return train_loader, val_loader, 'on_disk'


def calculate_metrics(y_true, y_pred_prob, threshold=0.5):
    """
    Calculate comprehensive metrics for binary classification.
    Main Metric: MCC (Matthews Correlation Coefficient)
    """
    y_pred = (y_pred_prob >= threshold).astype(int)
    
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0
    
    mcc = matthews_corrcoef(y_true, y_pred)
    
    try:
        auc_roc = roc_auc_score(y_true, y_pred_prob)
    except:
        auc_roc = 0.0
    
    try:
        precision_curve, recall_curve, _ = precision_recall_curve(y_true, y_pred_prob)
        auprc = auc(recall_curve, precision_curve)
    except:
        auprc = 0.0
    
    return {
        'accuracy': acc,
        'precision': prec,
        'recall': rec,
        'sensitivity': rec,
        'specificity': spec,
        'f1': f1,
        'mcc': mcc,
        'auc_roc': auc_roc,
        'auprc': auprc,
        'confusion_matrix': cm
    }


def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    all_preds = []
    all_labels = []
    
    for X, y in tqdm(loader, desc="Training", leave=False):
        X, y = X.to(device), y.to(device)
        
        optimizer.zero_grad()
        outputs = model(X)
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        preds = torch.sigmoid(outputs).detach().cpu().tolist()
        labels = y.cpu().tolist()
        all_preds.extend(preds)
        all_labels.extend(labels)
    
    return total_loss / len(loader), np.array(all_preds).flatten(), np.array(all_labels).flatten()


def validate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for X, y in tqdm(loader, desc="Validating", leave=False):
            X, y = X.to(device), y.to(device)
            
            outputs = model(X)
            loss = criterion(outputs, y)
            
            total_loss += loss.item()
            preds = torch.sigmoid(outputs).cpu().tolist()
            labels = y.cpu().tolist()
            all_preds.extend(preds)
            all_labels.extend(labels)
    
    return total_loss / len(loader), np.array(all_preds).flatten(), np.array(all_labels).flatten()


def rawecgnet_3fold_cv():
    set_seed()
    
    print("="*80)
    print("RawECGNet 3-FOLD CROSS-VALIDATION")
    print("   (Lead-Agnostic Training)")
    print("="*80)

    DATA_DIR = "/kaggle/input/shdbiridiaprocessed"
    H5_FILE = os.path.join(DATA_DIR, "afib_preprocessed.h5")
    SPLIT_FILE = os.path.join(DATA_DIR, "dataset_split.json")
    SAVE_DIR = "/kaggle/working/models/rawecgnet_3fold"
    
    # Set up resume environment
    INPUT_DATASET_PATH = "/kaggle/input/checkpoints"
    setup_resume_environment(INPUT_DATASET_PATH, SAVE_DIR)
    
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Device: {DEVICE}")
    
    BATCH_SIZE = 128
    LEARNING_RATE = 1e-4
    NUM_EPOCHS = 50
    EARLY_STOP_PATIENCE = 15
    
    print(f"\nLoading split from: {SPLIT_FILE}")
    with open(SPLIT_FILE, 'r') as f:
        split = json.load(f)
    
    train_patients = np.array(split['train_patients'])
    test_patients = split['test_patients']
    
    print(f"   Train patients: {len(train_patients)}")
    print(f"   Test patients: {len(test_patients)}")
    
    print(f"\nCalculating class weights...")
    with h5py.File(H5_FILE, 'r') as f:
        all_labels = f['y'][:]
        total_samples = len(all_labels)
    
    pos_count = np.sum(all_labels)
    neg_count = len(all_labels) - pos_count
    pos_weight = neg_count / pos_count
    print(f"   Total samples: {total_samples:,}")
    print(f"   Positive weight: {pos_weight:.2f}")
    
    progress_file = os.path.join(SAVE_DIR, "rawecgnet_progress.json")
    
    if os.path.exists(progress_file):
        print(f"\nFound existing progress file: {progress_file}")
        with open(progress_file, 'r') as f:
            progress = json.load(f)
        print(f"   Resuming from Fold {progress['current_fold']}, Epoch {progress['current_epoch']}")
    else:
        progress = {
            'current_fold': 1,
            'current_epoch': 1,
            'completed_folds': [],
            'fold_results': []
        }
        print(f"\nNo existing progress found. Starting fresh.")
    
    kfold = KFold(n_splits=3, shuffle=True, random_state=42)
    fold_results = progress['fold_results']
    
    for fold, (train_idx, val_idx) in enumerate(kfold.split(train_patients), 1):
        if fold < progress['current_fold']:
            print(f"\nSkipping Fold {fold} (already completed)")
            continue
        
        if fold in progress['completed_folds']:
            print(f"\nSkipping Fold {fold} (already completed)")
            continue
        
        print("\n" + "="*80)
        print(f"FOLD {fold}/3")
        print("="*80)
        
        fold_train_patients = train_patients[train_idx].tolist()
        fold_val_patients = train_patients[val_idx].tolist()
        
        print(f"   Fold {fold} train patients: {len(fold_train_patients)}")
        print(f"   Fold {fold} val patients: {len(fold_val_patients)}")
        
        train_loader, val_loader, mode_used = create_adaptive_datasets(
            H5_FILE, fold_train_patients, fold_val_patients, BATCH_SIZE
        )
        
        print(f"\nUsing mode: {mode_used.upper()}")
        print(f"Train batches: {len(train_loader)}")
        print(f"Val batches: {len(val_loader)}")
        
        model = RawECGNet(in_channels=1, num_classes=1, dropout=0.2).to(DEVICE)
        
        criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight]).to(DEVICE))
        optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
        
        model_save_path = os.path.join(SAVE_DIR, f"rawecgnet_fold{fold}_best.pth")
        checkpoint_path = os.path.join(SAVE_DIR, f"rawecgnet_fold{fold}_checkpoint.pth")
        
        best_val_mcc = -1
        best_epoch = 0
        epochs_without_improvement = 0
        start_epoch = 1
        early_stopped = False
        
        if fold == progress['current_fold'] and os.path.exists(checkpoint_path):
            print(f"\nLoading checkpoint from: {checkpoint_path}")
            checkpoint = torch.load(checkpoint_path, weights_only=False)
            model.load_state_dict(checkpoint['model_state_dict'])
            optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
            start_epoch = checkpoint['epoch'] + 1
            best_val_mcc = checkpoint['best_mcc']
            best_epoch = checkpoint['best_epoch']
            epochs_without_improvement = checkpoint['epochs_without_improvement']
            early_stopped = checkpoint.get('early_stopped', False)
            print(f"   Resuming from epoch {start_epoch}")
            print(f"   Best MCC so far: {best_val_mcc:.4f} (epoch {best_epoch})")
            if early_stopped:
                print(f"   Early stopping was already triggered - skipping to fold evaluation")
        
        if not early_stopped:
            for epoch in range(start_epoch, NUM_EPOCHS + 1):
                print(f"\nFold {fold} - Epoch {epoch}/{NUM_EPOCHS}")
                
                train_loss, train_preds, train_labels = train_epoch(
                    model, train_loader, criterion, optimizer, DEVICE)
                
                val_loss, val_preds, val_labels = validate(
                    model, val_loader, criterion, DEVICE)
                
                train_metrics = calculate_metrics(train_labels, train_preds)
                val_metrics = calculate_metrics(val_labels, val_preds)
                
                if val_metrics['mcc'] > best_val_mcc:
                    best_val_mcc = val_metrics['mcc']
                    best_epoch = epoch
                    epochs_without_improvement = 0
                    
                    torch.save({
                        'fold': fold,
                        'epoch': epoch,
                        'model_state_dict': model.state_dict(),
                        'optimizer_state_dict': optimizer.state_dict(),
                        'best_mcc': best_val_mcc,
                        'val_metrics': val_metrics,
                        'train_metrics': train_metrics,
                        'architecture': 'RawECGNet'
                    }, model_save_path)
                    
                    improvement_msg = "New best MCC! Model saved."
                else:
                    epochs_without_improvement += 1
                    improvement_msg = f"No improvement ({epochs_without_improvement}/{EARLY_STOP_PATIENCE})"
                
                print(f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
                print(f"Train MCC: {train_metrics['mcc']:.4f} | Val MCC: {val_metrics['mcc']:.4f}")
                print(f"{improvement_msg}")
                
                if epochs_without_improvement >= EARLY_STOP_PATIENCE:
                    early_stopped = True
                    print(f"\nEarly stopping triggered at epoch {epoch}")
                
                torch.save({
                    'fold': fold,
                    'epoch': epoch,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'best_mcc': best_val_mcc,
                    'best_epoch': best_epoch,
                    'epochs_without_improvement': epochs_without_improvement,
                    'early_stopped': early_stopped,
                    'val_metrics': val_metrics,
                    'train_metrics': train_metrics
                }, checkpoint_path)
                
                progress['current_fold'] = fold
                progress['current_epoch'] = epoch
                progress['early_stopped'] = early_stopped
                with open(progress_file, 'w') as f:
                    json.dump(progress, f, indent=2)
                
                if early_stopped:
                    break
        
        checkpoint = torch.load(model_save_path, weights_only=False)
        model.load_state_dict(checkpoint['model_state_dict'])
        
        _, val_preds, val_labels = validate(model, val_loader, criterion, DEVICE)
        final_metrics = calculate_metrics(val_labels, val_preds)
        
        print(f"\n" + "="*80)
        print(f"FOLD {fold} FINAL RESULTS")
        print("="*80)
        print(f"Best Epoch: {best_epoch}")
        
        print(f"\nConfusion Matrix:")
        cm = final_metrics['confusion_matrix']
        print(f"  TN: {cm[0,0]:6d}  |  FP: {cm[0,1]:6d}")
        print(f"  FN: {cm[1,0]:6d}  |  TP: {cm[1,1]:6d}")
        
        cm_plot_path = os.path.join(SAVE_DIR, f"confusion_matrix_fold{fold}.png")
        plot_confusion_matrix(cm, f"RawECGNet - Fold {fold} Confusion Matrix", cm_plot_path)
        
        print(f"\nClassification Metrics:")
        print(f"  Accuracy:    {final_metrics['accuracy']:.4f}")
        print(f"  Precision:   {final_metrics['precision']:.4f}")
        print(f"  Sensitivity: {final_metrics['sensitivity']:.4f}")
        print(f"  Specificity: {final_metrics['specificity']:.4f}")
        print(f"  F1-Score:    {final_metrics['f1']:.4f}")
        print(f"  MCC:         {final_metrics['mcc']:.4f}")
        print(f"  AUC-ROC:     {final_metrics['auc_roc']:.4f}")
        print(f"  AUPRC:       {final_metrics['auprc']:.4f}")
        
        # Convert confusion matrix to list for JSON serialization
        final_metrics_serializable = final_metrics.copy()
        final_metrics_serializable['confusion_matrix'] = final_metrics['confusion_matrix'].tolist()
        
        fold_results.append(final_metrics_serializable)
        
        progress['completed_folds'].append(fold)
        progress['fold_results'] = fold_results
        progress['current_fold'] = fold + 1
        progress['current_epoch'] = 1
        with open(progress_file, 'w') as f:
            json.dump(progress, f, indent=2)
        
        if os.path.exists(checkpoint_path):
            os.remove(checkpoint_path)
        
        print(f"\nFold {fold} completed and saved to progress.")
    
    print("\n" + "="*80)
    print("3-FOLD CROSS-VALIDATION SUMMARY")
    print("="*80)
    
    avg_cm = np.mean([r['confusion_matrix'] for r in fold_results], axis=0)
    print(f"\nAverage Confusion Matrix:")
    print(f"  TN: {avg_cm[0,0]:8.1f}  |  FP: {avg_cm[0,1]:8.1f}")
    print(f"  FN: {avg_cm[1,0]:8.1f}  |  TP: {avg_cm[1,1]:8.1f}")
    
    avg_cm_plot_path = os.path.join(SAVE_DIR, "confusion_matrix_average.png")
    plot_confusion_matrix(avg_cm.astype(int), "RawECGNet - Average Confusion Matrix (3-Fold CV)", avg_cm_plot_path)
    
    print(f"\nAverage Classification Metrics (across 3 folds):")
    for metric in ['accuracy', 'precision', 'sensitivity', 'specificity', 'f1', 'mcc', 'auc_roc', 'auprc']:
        values = [r[metric] for r in fold_results]
        mean_val = np.mean(values)
        std_val = np.std(values)
        print(f"  {metric.capitalize():12s}: {mean_val:.4f} ± {std_val:.4f}")
    
    print("\n" + "="*80)
    print(f"Models saved in: {SAVE_DIR}")
    print("="*80)

# --- FINAL TRAINING START ---
    print("\n" + "="*40 + "\nFINAL TRAINING (Full Train -> Test)\n" + "="*40)

    # 1. Setup Datasets (Train + Test)
    final_train_loader, final_test_loader, _ = create_adaptive_datasets(
        H5_FILE, train_patients, test_patients, BATCH_SIZE
    )

    # 2. Reset Model & Optimizer
    final_model = RawECGNet(in_channels=1, num_classes=1, dropout=0.2).to(DEVICE)
    optimizer = torch.optim.Adam(final_model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight]).to(DEVICE))
    
    final_save_path = os.path.join(SAVE_DIR, "rawecgnet_FINAL.pth")
    best_mcc = -1
    no_improve = 0

    # 3. Train Loop
    for epoch in range(1, NUM_EPOCHS + 1):
        print(f"\nFinal Epoch {epoch}/{NUM_EPOCHS}")
        
        train_loss, train_p, train_l = train_epoch(final_model, final_train_loader, criterion, optimizer, DEVICE)
        test_loss, test_p, test_l = validate(final_model, final_test_loader, criterion, DEVICE)
        
        train_mcc = matthews_corrcoef(train_l, (train_p > 0.5).astype(int))
        test_metrics = calculate_metrics(test_l, test_p)
        
        print(f"Loss: {train_loss:.4f} (Train) | {test_loss:.4f} (Test)")
        print(f"MCC:  {train_mcc:.4f} (Train) | {test_metrics['mcc']:.4f} (Test)")
        
        if test_metrics['mcc'] > best_mcc:
            best_mcc = test_metrics['mcc']
            no_improve = 0
            torch.save({
                'epoch': epoch,
                'model': final_model.state_dict(),
                'metrics': test_metrics
            }, final_save_path)
            print(">> Saved New Best Model")
        else:
            no_improve += 1
            print(f">> No improve ({no_improve}/{EARLY_STOP_PATIENCE})")
            
        if no_improve >= EARLY_STOP_PATIENCE:
            print("Stopping Early.")
            break

    # 4. Final Evaluation
    print("\n" + "="*40 + "\nFINAL TEST REPORT\n" + "="*40)
    checkpoint = torch.load(final_save_path)
    final_model.load_state_dict(checkpoint['model'])
    
    _, final_p, final_l = validate(final_model, final_test_loader, criterion, DEVICE)
    final_metrics = calculate_metrics(final_l, final_p)
    
    plot_confusion_matrix(final_metrics['confusion_matrix'], "RawECGNet - Final Test Matrix", os.path.join(SAVE_DIR, "cm_final.png"))
    
    print(f"\nClassification Metrics:")
    print(f"  Accuracy:    {final_metrics['accuracy']:.4f}")
    print(f"  Precision:   {final_metrics['precision']:.4f}")
    print(f"  Sensitivity: {final_metrics['sensitivity']:.4f}")
    print(f"  Specificity: {final_metrics['specificity']:.4f}")
    print(f"  F1-Score:    {final_metrics['f1']:.4f}")
    print(f"  MCC:         {final_metrics['mcc']:.4f}")
    print(f"  AUC-ROC:     {final_metrics['auc_roc']:.4f}")
    print(f"  AUPRC:       {final_metrics['auprc']:.4f}")
    
    # Save final metrics to JSON
    with open(os.path.join(SAVE_DIR, "final_test_results.json"), 'w') as f:
        serializable_mets = {k: (v.tolist() if isinstance(v, np.ndarray) else v) for k, v in final_metrics.items()}
        json.dump(serializable_mets, f, indent=2)

if __name__ == "__main__":
    rawecgnet_3fold_cv()

## CTRhythm (Liang et al.)

In [ ]:
import os
import json
import h5py
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, matthews_corrcoef,
                             confusion_matrix, precision_recall_curve, auc)
from sklearn.model_selection import KFold
from tqdm import tqdm
import time
from datetime import timedelta
import math
import matplotlib.pyplot as plt
import warnings
import gc
import random
import threading
import shutil
warnings.filterwarnings('ignore')

SEED = 198

def seed_worker(worker_id):
    worker_seed = SEED + worker_id
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def setup_resume_environment(input_root, save_dir):
    """
    Copy checkpoint files from read-only input dataset to writable save directory.

    Args:
        input_root: Path to read-only input dataset directory
        save_dir: Path to writable save directory
    """
    print(f"\nSetting up resume environment...")
    print(f"   Input dataset path: {input_root}")
    print(f"   Save directory: {save_dir}")

    # Create save directory if it doesn't exist
    os.makedirs(save_dir, exist_ok=True)
    print(f"   Save directory created/verified")

    # Check if input dataset exists
    if not os.path.exists(input_root):
        print(f"   Input dataset not found - starting fresh training")
        return

    # Copy all .json and .pth files from input to save directory
    copied_files = 0
    for filename in os.listdir(input_root):
        if filename.endswith('.json') or filename.endswith('.pth'):
            src_path = os.path.join(input_root, filename)
            dst_path = os.path.join(save_dir, filename)

            # Only copy if destination doesn't exist or is older
            if not os.path.exists(dst_path):
                shutil.copy2(src_path, dst_path)
                print(f"   Copied: {filename}")
                copied_files += 1

    if copied_files == 0:
        print(f"   No checkpoint files found to copy - starting fresh training")
    else:
        print(f"   Successfully copied {copied_files} file(s)")

def plot_confusion_matrix(cm, title, save_path):
    """Plot and save confusion matrix"""
    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(cm, interpolation='nearest', cmap='Blues')
    ax.figure.colorbar(im, ax=ax, label='Count')

    classes = ['Normal', 'AFib']
    ax.set(xticks=np.arange(cm.shape[1]),
           yticks=np.arange(cm.shape[0]),
           xticklabels=classes, yticklabels=classes,
           title=title,
           ylabel='True Label',
           xlabel='Predicted Label')

    ax.set_title(title, fontsize=14, fontweight='bold', pad=20)
    ax.set_ylabel('True Label', fontsize=12)
    ax.set_xlabel('Predicted Label', fontsize=12)

    thresh = cm.max() / 2.
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, format(cm[i, j], 'd'),
                   ha="center", va="center",
                   color="white" if cm[i, j] > thresh else "black",
                   fontsize=14, fontweight='bold')

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"   Confusion matrix plot saved: {save_path}")


class ConvBlock(nn.Module):
    """Basic Convolutional Block with BatchNorm and ReLU"""
    def __init__(self, channel_in, channel, stride=1):
        super().__init__()
        self.conv = nn.Conv1d(channel_in, channel, kernel_size=3, padding=1, stride=stride)
        self.norm = nn.BatchNorm1d(channel)
        self.activate = nn.ReLU()

    def forward(self, x):
        x = self.conv(x)
        x = self.norm(x)
        x = self.activate(x)
        return x


class ResidualBlock(nn.Module):
    """Residual Block with skip connection"""
    def __init__(self, channel_in=32, channel=32, stride=1, dropout=0.3):
        super().__init__()
        self.stride = stride
        self.dropout = nn.Dropout(p=dropout, inplace=False)
        self.main = nn.Sequential(
            ConvBlock(channel_in, channel, stride=stride),
            ConvBlock(channel, channel),
            ConvBlock(channel, channel),
        )
        self.skip = None
        if self.stride == 2:
            self.skip = nn.Conv1d(channel_in, channel, kernel_size=3, padding=1, stride=stride)

    def forward(self, x):
        x = self.dropout(x)
        y = self.main(x)
        if self.stride == 2:
            x = self.skip(x)
        x = y + x
        return x


class CNNModule(nn.Module):
    """CNN Module for feature extraction and downsampling"""
    def __init__(self, channel_in=1, downsample_depth=5, main_depth=1, dropout=0.3):
        super().__init__()

        self.downsample_layers = nn.Sequential(
            ResidualBlock(channel_in, 16, 2, dropout),
            ResidualBlock(16, 32, 2, dropout),
        )
        self.main = nn.Sequential(
            ResidualBlock(dropout=dropout)
        )
        if downsample_depth > 2:
            for i in range(downsample_depth - 2):
                self.downsample_layers.append(ResidualBlock(32, 32, 2, dropout=dropout))
        if main_depth > 1:
            for i in range(main_depth - 1):
                self.downsample_layers.append(ResidualBlock(32, 32, 1, dropout=dropout))

    def forward(self, x):
        x = self.downsample_layers(x)
        x = self.main(x)
        return x


class PositionalEncoding(nn.Module):
    def __init__(self, d_in, d_model, dropout=0.1, max_len=10000):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * -(math.log(10000.0) / d_model))
        pe = torch.zeros(max_len, 1, d_model)
        pe[:, 0, 0::2] = torch.sin(position * div_term)
        pe[:, 0, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe)
        self.linear = nn.Linear(d_in, d_model)

    def forward(self, x):
        # x shape expected: [seq_len, batch_size, d_in]
        x = self.linear(x)  # [seq_len, batch_size, d_model]
        # self.pe[:x.size(0)] gets the first seq_len elements. Shape: [seq_len, 1, d_model]
        # Broadcasting adds it to all batches
        x = x + self.pe[:x.size(0)] 
        x = self.dropout(x)
        return x


class TransformerEncoder(nn.Module):
    """Transformer Encoder Module"""
    def __init__(self, d_model, nhead, num_layers, dim_feedforward, dropout=0.1):
        super().__init__()
        encoder_layer = nn.TransformerEncoderLayer(d_model, nhead, dim_feedforward, dropout, batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers)

    def forward(self, x):
        x = self.transformer_encoder(x)
        return x


class CTRhythm(nn.Module):
    """
    CTRhythm Architecture - Lead-Agnostic Binary AFib Detection
    Follows original CTRhythm structure with binary classification head
    Input: [batch, 1, seq_len]
    Output: [batch, 1] (logits for binary classification)
    """
    def __init__(self, init_channels=1, d_model=32, nhead=8, num_layers=6,
                 dim_feedforward=256, num_classes=1, dropout=0.1):
        super().__init__()
        self.CNN = CNNModule(channel_in=init_channels, downsample_depth=5, main_depth=1, dropout=0.3)
        self.pos_encoding = PositionalEncoding(32, d_model, dropout)
        self.transformer_encoder = TransformerEncoder(d_model, nhead, num_layers, dim_feedforward, dropout=dropout)
        self.classification = nn.Linear(d_model, num_classes)

    def forward_feature(self, x):
        """Extract features without classification"""
        x = self.CNN(x)  # [batch, d_model, seq_len]
        x = x.permute(2, 0, 1)  # [seq_len, batch, d_model]
        x = self.pos_encoding(x)  # [seq_len, batch, d_model]
        x = x.permute(1, 0, 2)  # [batch, seq_len, d_model]
        x = self.transformer_encoder(x)  # [batch, seq_len, d_model]
        x = torch.mean(x, dim=1)  # [batch, d_model]
        return x

    def forward(self, x):
        """Forward pass - expects input shape [batch, 1, seq_len]"""
        x = self.forward_feature(x)
        x = self.classification(x)
        return x


class LeadAgnosticDataset(Dataset):
    """
    Lead-Agnostic Dataset for Binary AFib Detection
    Returns data in shape [1, seq_len] to match CTRhythm input expectations
    """
    def __init__(self, h5_path, patient_ids, augment_with_both_leads=True):
        self.h5_path = h5_path
        self.patient_ids = set(patient_ids)
        self.augment_with_both_leads = augment_with_both_leads

        print(f"   Scanning HDF5 for {len(patient_ids)} patients (Lead-Agnostic Mode)...")
        with h5py.File(h5_path, 'r') as f:
            all_groups = f['groups'][:]
            all_groups = [g.decode('utf-8') if isinstance(g, bytes) else g
                         for g in all_groups]

            self.indices = [i for i, g in enumerate(all_groups)
                           if g in self.patient_ids]

        if self.augment_with_both_leads:
            print(f"   Dataset ready: {len(self.indices)} samples × 2 leads = {len(self.indices)*2} total samples")
        else:
            print(f"   Dataset ready: {len(self.indices)} samples (random lead per sample)")

    def __len__(self):
        if self.augment_with_both_leads:
            return len(self.indices) * 2
        return len(self.indices)

    def __getitem__(self, idx):
        if self.augment_with_both_leads:
            actual_idx = self.indices[idx % len(self.indices)]
            lead_idx = 0 if idx < len(self.indices) else 1
        else:
            actual_idx = self.indices[idx]
            lead_idx = np.random.randint(0, 2)

        with h5py.File(self.h5_path, 'r') as f:
            X = f['X'][actual_idx]
            y = f['y'][actual_idx]

        # Extract single lead: [seq_len, 1]
        X_lead = X[:, lead_idx:lead_idx+1]

        # Transpose to match CTRhythm input: [1, seq_len]
        X_lead = X_lead.T

        return torch.from_numpy(X_lead).float(), torch.tensor([y], dtype=torch.float32)


class RAMEfficientDataset(Dataset):
    """
    RAM-efficient dataset with auto-detection
    Returns data in shape [1, seq_len] to match CTRhythm input expectations
    """
    def __init__(self, h5_path, patient_ids, augment_with_both_leads=True,
                 preload_mode='auto', use_float16=True, max_ram_gb=20):
        self.h5_path = h5_path
        self.patient_ids = set(patient_ids)
        self.augment_with_both_leads = augment_with_both_leads
        self.use_float16 = use_float16
        self.max_ram_gb = max_ram_gb

        print(f"   Initializing RAM-Efficient Dataset...")
        print(f"      Mode: {preload_mode} | Float16: {use_float16} | Max RAM: {max_ram_gb} GB")

        with h5py.File(h5_path, 'r') as f:
            all_groups = f['groups'][:]
            all_groups = [g.decode('utf-8') if isinstance(g, bytes) else g
                         for g in all_groups]
            self.indices = [i for i, g in enumerate(all_groups)
                           if g in self.patient_ids]
            self.sample_shape = f['X'].shape[1:]

        print(f"      Found {len(self.indices)} matching samples")

        bytes_per_sample = np.prod(self.sample_shape) * (2 if use_float16 else 4)
        total_gb = (len(self.indices) * bytes_per_sample) / (1024**3)
        print(f"      Estimated RAM: {total_gb:.2f} GB")

        if preload_mode == 'auto':
            if total_gb < max_ram_gb * 0.7:
                preload_mode = 'full'
                print(f"      Auto-selected: FULL preload")
            elif total_gb < max_ram_gb * 1.2:
                preload_mode = 'chunk'
                print(f"      Auto-selected: CHUNK mode")
            else:
                preload_mode = 'lazy'
                print(f"      Auto-selected: LAZY mode")

        self.preload_mode = preload_mode
        self.X_cache = None
        self.y_cache = None
        self.chunk_cache = {}
        self.chunk_size = 10000

        if preload_mode == 'full':
            self._preload_full()

        if self.augment_with_both_leads:
            print(f"      Dataset ready: {len(self.indices)} samples × 2 leads = {len(self.indices)*2} total samples")
        else:
            print(f"      Dataset ready: {len(self.indices)} samples (random lead per sample)")

    def _preload_full(self):
        print(f"      Preloading into RAM...")
        dtype = np.float16 if self.use_float16 else np.float32

        self.X_cache = np.empty((len(self.indices), *self.sample_shape), dtype=dtype)
        self.y_cache = np.empty(len(self.indices), dtype=np.int8)

        batch_size = 50000
        with h5py.File(self.h5_path, 'r') as f:
            for i in range(0, len(self.indices), batch_size):
                end_idx = min(i + batch_size, len(self.indices))
                batch_indices = self.indices[i:end_idx]

                X_batch = f['X'][batch_indices]
                y_batch = f['y'][batch_indices]

                self.X_cache[i:end_idx] = X_batch.astype(dtype) if self.use_float16 else X_batch
                self.y_cache[i:end_idx] = y_batch

                if end_idx % 100000 == 0:
                    print(f"         {end_idx:,}/{len(self.indices):,}")

        print(f"      Preload complete!")
        gc.collect()

    def _load_chunk(self, chunk_id):
        if chunk_id in self.chunk_cache:
            returnr

        start_idx = chunk_id * self.chunk_size
        end_idx = min(start_idx + self.chunk_size, len(self.indices))
        chunk_indices = self.indices[start_idx:end_idx]

        dtype = np.float16 if self.use_float16 else np.float32

        with h5py.File(self.h5_path, 'r') as f:
            X_chunk = f['X'][chunk_indices]
            y_chunk = f['y'][chunk_indices]

            if self.use_float16:
                X_chunk = X_chunk.astype(np.float16)

            self.chunk_cache[chunk_id] = (X_chunk, y_chunk)

        if len(self.chunk_cache) > 3:
            oldest = min(self.chunk_cache.keys())
            del self.chunk_cache[oldest]
            gc.collect()

    def __len__(self):
        return len(self.indices) * 2 if self.augment_with_both_leads else len(self.indices)

    def __getitem__(self, idx):
        if self.augment_with_both_leads:
            actual_idx = idx % len(self.indices)
            lead_idx = 0 if idx < len(self.indices) else 1
        else:
            actual_idx = idx
            lead_idx = np.random.randint(0, 2)

        if self.preload_mode == 'full':
            X = self.X_cache[actual_idx]
            y = self.y_cache[actual_idx]
        elif self.preload_mode == 'chunk':
            chunk_id = actual_idx // self.chunk_size
            self._load_chunk(chunk_id)
            local_idx = actual_idx % self.chunk_size
            X, y_chunk = self.chunk_cache[chunk_id]
            X = X[local_idx]
            y = y_chunk[local_idx]
        else:
            with h5py.File(self.h5_path, 'r') as f:
                global_idx = self.indices[actual_idx]
                X = f['X'][global_idx]
                y = f['y'][global_idx]

        if self.use_float16 and isinstance(X, np.ndarray):
            X = X.astype(np.float32)

        # Extract single lead: [seq_len, 1]
        X_lead = X[:, lead_idx:lead_idx+1]

        # Transpose to match CTRhythm input: [1, seq_len]
        X_lead = X_lead.T

        return torch.from_numpy(X_lead).float(), torch.tensor([y], dtype=torch.float32)


def test_dataloader_with_timeout(loader, timeout_seconds=30):
    """Test if dataloader works within timeout"""
    print(f"   Testing dataloader (timeout: {timeout_seconds}s)...")

    result = {'success': False, 'error': None}

    def load_batch():
        try:
            iterator = iter(loader)
            X, y = next(iterator)
            result['success'] = True
            result['batch_shape'] = X.shape
        except Exception as e:
            result['error'] = str(e)

    thread = threading.Thread(target=load_batch)
    thread.daemon = True
    thread.start()
    thread.join(timeout=timeout_seconds)

    if thread.is_alive():
        print(f"   DataLoader test TIMED OUT after {timeout_seconds}s")
        return False

    if result['success']:
        print(f"   DataLoader test PASSED (batch shape: {result['batch_shape']})")
        return True
    else:
        print(f"   DataLoader test FAILED: {result['error']}")
        return False


def create_adaptive_datasets(h5_path, train_patients, val_patients, batch_size=128):
    """Adaptive dataset creation with automatic fallback"""
    print("\n" + "="*80)
    print("ADAPTIVE DATASET CREATION")
    print("="*80)

    print("\nAttempting RAM-Efficient Mode...")
    try:
        train_dataset = RAMEfficientDataset(
            h5_path, train_patients,
            augment_with_both_leads=True,
            preload_mode='auto',
            use_float16=True,
            max_ram_gb=20
        )

        val_dataset = RAMEfficientDataset(
            h5_path, val_patients,
            augment_with_both_leads=True,
            preload_mode='auto',
            use_float16=True,
            max_ram_gb=20
        )

        g = torch.Generator()
        g.manual_seed(SEED)

        train_loader = DataLoader(
            train_dataset,
            batch_size=batch_size,
            shuffle=True,
            num_workers=2,
            worker_init_fn=seed_worker,
            generator=g,
            pin_memory=True,
            persistent_workers=True
        )

        if test_dataloader_with_timeout(train_loader, timeout_seconds=30):
            print("\nRAM-Efficient Mode SUCCESSFUL - Using with 2 workers")

            val_loader = DataLoader(
                val_dataset,
                batch_size=batch_size,
                shuffle=False,
                num_workers=2,
                pin_memory=True,
                persistent_workers=True
            )

            return train_loader, val_loader, 'ram_efficient'
        else:
            print("\nRAM-Efficient Mode got stuck - Falling back...")
            del train_loader, train_dataset, val_dataset
            gc.collect()

    except Exception as e:
        print(f"\nRAM-Efficient Mode failed: {e}")
        print("   Falling back to on-disk loading...")

    print("\nUsing On-Disk Loading Mode (Original)...")
    train_dataset = LeadAgnosticDataset(h5_path, train_patients, augment_with_both_leads=True)
    val_dataset = LeadAgnosticDataset(h5_path, val_patients, augment_with_both_leads=True)

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0
    )

    print("On-Disk Mode ready (num_workers=0)")
    return train_loader, val_loader, 'on_disk'


def calculate_metrics(y_true, y_pred_prob, threshold=0.5):
    """
    Calculate comprehensive metrics for binary classification.
    Main Metric: MCC (Matthews Correlation Coefficient)
    """
    y_pred = (y_pred_prob >= threshold).astype(int)

    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0

    mcc = matthews_corrcoef(y_true, y_pred)

    try:
        auc_roc = roc_auc_score(y_true, y_pred_prob)
    except:
        auc_roc = 0.0

    try:
        precision_curve, recall_curve, _ = precision_recall_curve(y_true, y_pred_prob)
        auprc = auc(recall_curve, precision_curve)
    except:
        auprc = 0.0

    return {
        'accuracy': acc,
        'precision': prec,
        'recall': rec,
        'sensitivity': rec,
        'specificity': spec,
        'f1': f1,
        'mcc': mcc,
        'auc_roc': auc_roc,
        'auprc': auprc,
        'confusion_matrix': cm
    }


def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    all_preds = []
    all_labels = []

    for X, y in tqdm(loader, desc="Training", leave=False):
        X, y = X.to(device), y.to(device)

        optimizer.zero_grad()
        outputs = model(X)
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds = torch.sigmoid(outputs).detach().cpu().tolist()
        labels = y.cpu().tolist()
        all_preds.extend(preds)
        all_labels.extend(labels)

    return total_loss / len(loader), np.array(all_preds).flatten(), np.array(all_labels).flatten()


def validate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for X, y in tqdm(loader, desc="Validating", leave=False):
            X, y = X.to(device), y.to(device)

            outputs = model(X)
            loss = criterion(outputs, y)

            total_loss += loss.item()
            preds = torch.sigmoid(outputs).cpu().tolist()
            labels = y.cpu().tolist()
            all_preds.extend(preds)
            all_labels.extend(labels)

    return total_loss / len(loader), np.array(all_preds).flatten(), np.array(all_labels).flatten()


def ctrhythm_3fold_cv():
    set_seed()

    print("="*80)
    print("CTRhythm 3-FOLD CROSS-VALIDATION")
    print("   (Lead-Agnostic Binary AFib Detection)")
    print("="*80)

    DATA_DIR = "/kaggle/input/shdbiridiaprocessed"
    H5_FILE = os.path.join(DATA_DIR, "afib_preprocessed.h5")
    SPLIT_FILE = os.path.join(DATA_DIR, "dataset_split.json")
    SAVE_DIR = "/kaggle/working/models/ctrhythm_3fold"

    # Set up resume environment
    INPUT_DATASET_PATH = "/kaggle/input/datasets/sheianne/checkpoints"
    setup_resume_environment(INPUT_DATASET_PATH, SAVE_DIR)

    os.makedirs(SAVE_DIR, exist_ok=True)

    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Device: {DEVICE}")

    BATCH_SIZE = 128
    LEARNING_RATE = 1e-4
    NUM_EPOCHS = 50
    EARLY_STOP_PATIENCE = 15

    D_MODEL = 32
    NHEAD = 8
    NUM_LAYERS = 6
    DIM_FEEDFORWARD = 256
    DROPOUT = 0.1

    print(f"\nLoading split from: {SPLIT_FILE}")
    with open(SPLIT_FILE, 'r') as f:
        split = json.load(f)

    train_patients = np.array(split['train_patients'])
    test_patients = split['test_patients']

    print(f"   Train patients: {len(train_patients)}")
    print(f"   Test patients: {len(test_patients)}")

    print(f"\nCalculating class weights...")
    with h5py.File(H5_FILE, 'r') as f:
        all_labels = f['y'][:]
        total_samples = len(all_labels)

    pos_count = np.sum(all_labels)
    neg_count = len(all_labels) - pos_count
    pos_weight = neg_count / pos_count
    print(f"   Total samples: {total_samples:,}")
    print(f"   Positive weight: {pos_weight:.2f}")

    progress_file = os.path.join(SAVE_DIR, "ctrhythm_progress.json")

    if os.path.exists(progress_file):
        print(f"\nFound existing progress file: {progress_file}")
        with open(progress_file, 'r') as f:
            progress = json.load(f)
        print(f"   Resuming from Fold {progress['current_fold']}, Epoch {progress['current_epoch']}")
    else:
        progress = {
            'current_fold': 1,
            'current_epoch': 1,
            'completed_folds': [],
            'fold_results': []
        }
        print(f"\nNo existing progress found. Starting fresh.")

    kfold = KFold(n_splits=3, shuffle=True, random_state=42)
    fold_results = progress['fold_results']

    for fold, (train_idx, val_idx) in enumerate(kfold.split(train_patients), 1):
        if fold < progress['current_fold']:
            print(f"\nSkipping Fold {fold} (already completed)")
            continue

        if fold in progress['completed_folds']:
            print(f"\nSkipping Fold {fold} (already completed)")
            continue

        print("\n" + "="*80)
        print(f"FOLD {fold}/3")
        print("="*80)

        fold_train_patients = train_patients[train_idx].tolist()
        fold_val_patients = train_patients[val_idx].tolist()

        print(f"   Fold {fold} train patients: {len(fold_train_patients)}")
        print(f"   Fold {fold} val patients: {len(fold_val_patients)}")

        train_loader, val_loader, mode_used = create_adaptive_datasets(
            H5_FILE, fold_train_patients, fold_val_patients, BATCH_SIZE
        )

        print(f"\nUsing mode: {mode_used.upper()}")
        print(f"Train batches: {len(train_loader)}")
        print(f"Val batches: {len(val_loader)}")

        model = CTRhythm(init_channels=1, d_model=D_MODEL, nhead=NHEAD,
                        num_layers=NUM_LAYERS, dim_feedforward=DIM_FEEDFORWARD,
                        num_classes=1, dropout=DROPOUT).to(DEVICE)

        criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight]).to(DEVICE))
        optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)

        model_save_path = os.path.join(SAVE_DIR, f"ctrhythm_fold{fold}_best.pth")
        checkpoint_path = os.path.join(SAVE_DIR, f"ctrhythm_fold{fold}_checkpoint.pth")

        best_val_mcc = -1
        best_epoch = 0
        epochs_without_improvement = 0
        start_epoch = 1
        early_stopped = False

        if fold == progress['current_fold'] and os.path.exists(checkpoint_path):
            print(f"\nLoading checkpoint from: {checkpoint_path}")
            checkpoint = torch.load(checkpoint_path, weights_only=False)
            model.load_state_dict(checkpoint['model_state_dict'])
            optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
            start_epoch = checkpoint['epoch'] + 1
            best_val_mcc = checkpoint['best_mcc']
            best_epoch = checkpoint['best_epoch']
            epochs_without_improvement = checkpoint['epochs_without_improvement']
            early_stopped = checkpoint.get('early_stopped', False)
            print(f"   Resuming from epoch {start_epoch}")
            print(f"   Best MCC so far: {best_val_mcc:.4f} (epoch {best_epoch})")
            if early_stopped:
                print(f"   Early stopping was already triggered - skipping to fold evaluation")

        if not early_stopped:
            for epoch in range(start_epoch, NUM_EPOCHS + 1):
                print(f"\nFold {fold} - Epoch {epoch}/{NUM_EPOCHS}")

                train_loss, train_preds, train_labels = train_epoch(
                    model, train_loader, criterion, optimizer, DEVICE)

                val_loss, val_preds, val_labels = validate(
                    model, val_loader, criterion, DEVICE)

                train_metrics = calculate_metrics(train_labels, train_preds)
                val_metrics = calculate_metrics(val_labels, val_preds)

                if val_metrics['mcc'] > best_val_mcc:
                    best_val_mcc = val_metrics['mcc']
                    best_epoch = epoch
                    epochs_without_improvement = 0

                    torch.save({
                        'fold': fold,
                        'epoch': epoch,
                        'model_state_dict': model.state_dict(),
                        'optimizer_state_dict': optimizer.state_dict(),
                        'best_mcc': best_val_mcc,
                        'val_metrics': val_metrics,
                        'train_metrics': train_metrics,
                        'architecture': 'CTRhythm'
                    }, model_save_path)

                    improvement_msg = "✓ New best MCC! Model saved."
                else:
                    epochs_without_improvement += 1
                    improvement_msg = f"No improvement ({epochs_without_improvement}/{EARLY_STOP_PATIENCE})"

                print(f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
                print(f"Train MCC: {train_metrics['mcc']:.4f} | Val MCC: {val_metrics['mcc']:.4f}")
                print(f"{improvement_msg}")

                if epochs_without_improvement >= EARLY_STOP_PATIENCE:
                    early_stopped = True
                    print(f"\nEarly stopping triggered at epoch {epoch}")

                torch.save({
                    'fold': fold,
                    'epoch': epoch,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'best_mcc': best_val_mcc,
                    'best_epoch': best_epoch,
                    'epochs_without_improvement': epochs_without_improvement,
                    'early_stopped': early_stopped,
                    'val_metrics': val_metrics,
                    'train_metrics': train_metrics
                }, checkpoint_path)

                progress['current_fold'] = fold
                progress['current_epoch'] = epoch
                progress['early_stopped'] = early_stopped
                with open(progress_file, 'w') as f:
                    json.dump(progress, f, indent=2)

                if early_stopped:
                    break

        checkpoint = torch.load(model_save_path, weights_only=False)
        model.load_state_dict(checkpoint['model_state_dict'])

        _, val_preds, val_labels = validate(model, val_loader, criterion, DEVICE)
        final_metrics = calculate_metrics(val_labels, val_preds)

        print(f"\n" + "="*80)
        print(f"FOLD {fold} FINAL RESULTS")
        print("="*80)
        print(f"Best Epoch: {best_epoch}")

        print(f"\nConfusion Matrix:")
        cm = final_metrics['confusion_matrix']
        print(f"  TN: {cm[0,0]:6d}  |  FP: {cm[0,1]:6d}")
        print(f"  FN: {cm[1,0]:6d}  |  TP: {cm[1,1]:6d}")

        cm_plot_path = os.path.join(SAVE_DIR, f"confusion_matrix_fold{fold}.png")
        plot_confusion_matrix(cm, f"CTRhythm - Fold {fold} Confusion Matrix", cm_plot_path)

        print(f"\nClassification Metrics:")
        print(f"  Accuracy:    {final_metrics['accuracy']:.4f}")
        print(f"  Precision:   {final_metrics['precision']:.4f}")
        print(f"  Sensitivity: {final_metrics['sensitivity']:.4f}")
        print(f"  Specificity: {final_metrics['specificity']:.4f}")
        print(f"  F1-Score:    {final_metrics['f1']:.4f}")
        print(f"  MCC:         {final_metrics['mcc']:.4f}")
        print(f"  AUC-ROC:     {final_metrics['auc_roc']:.4f}")
        print(f"  AUPRC:       {final_metrics['auprc']:.4f}")

        final_metrics_serializable = final_metrics.copy()
        final_metrics_serializable['confusion_matrix'] = final_metrics['confusion_matrix'].tolist()

        fold_results.append(final_metrics_serializable)

        progress['completed_folds'].append(fold)
        progress['fold_results'] = fold_results
        progress['current_fold'] = fold + 1
        progress['current_epoch'] = 1
        with open(progress_file, 'w') as f:
            json.dump(progress, f, indent=2)

        if os.path.exists(checkpoint_path):
            os.remove(checkpoint_path)

        print(f"\nFold {fold} completed and saved to progress.")

    print("\n" + "="*80)
    print("3-FOLD CROSS-VALIDATION SUMMARY")
    print("="*80)

    avg_cm = np.mean([np.array(r['confusion_matrix']) for r in fold_results], axis=0)
    print(f"\nAverage Confusion Matrix:")
    print(f"  TN: {avg_cm[0,0]:8.1f}  |  FP: {avg_cm[0,1]:8.1f}")
    print(f"  FN: {avg_cm[1,0]:8.1f}  |  TP: {avg_cm[1,1]:8.1f}")

    avg_cm_plot_path = os.path.join(SAVE_DIR, "confusion_matrix_average.png")
    plot_confusion_matrix(avg_cm.astype(int), "CTRhythm - Average Confusion Matrix (3-Fold CV)", avg_cm_plot_path)

    print(f"\nAverage Classification Metrics (across 3 folds):")
    for metric in ['accuracy', 'precision', 'sensitivity', 'specificity', 'f1', 'mcc', 'auc_roc', 'auprc']:
        values = [r[metric] for r in fold_results]
        mean_val = np.mean(values)
        std_val = np.std(values)
        print(f"  {metric.capitalize():12s}: {mean_val:.4f} ± {std_val:.4f}")

# FINAL TRAINING (Full Train -> Test)
    print("\n" + "="*80)
    print("FINAL TRAINING (Robust Resume)")
    print("="*80)

    final_train_loader, final_test_loader, _ = create_adaptive_datasets(
        H5_FILE, train_patients.tolist(), test_patients, BATCH_SIZE
    )

    final_model = CTRhythm(init_channels=1, d_model=D_MODEL, nhead=NHEAD,
                           num_layers=NUM_LAYERS, dim_feedforward=DIM_FEEDFORWARD,
                           num_classes=1, dropout=DROPOUT).to(DEVICE)

    optimizer = torch.optim.AdamW(final_model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight]).to(DEVICE))

    final_save_path = os.path.join(SAVE_DIR, "ctrhythm_FINAL.pth")
    progress_file = os.path.join(SAVE_DIR, "ctrhythm_progress.json") # Matched to CV loop JSON

    best_mcc = -1
    no_improve = 0
    start_epoch = 1

    # --- RESUME LOGIC ---
    if os.path.exists(final_save_path):
        print(f"\nFound existing final checkpoint: {final_save_path}")
        try:
            checkpoint = torch.load(final_save_path, weights_only=False)
            final_model.load_state_dict(checkpoint['model_state_dict']) # Matches CTRhythm save format

            if 'epoch' in checkpoint:
                start_epoch = checkpoint['epoch'] + 1

            if 'test_metrics' in checkpoint and 'mcc' in checkpoint['test_metrics']:
                best_mcc = checkpoint['test_metrics']['mcc']

            print(f"   Resuming from Epoch {start_epoch}")
            print(f"   Current Best MCC: {best_mcc:.4f}")

            # --- JSON PATCHING ---
            if os.path.exists(progress_file):
                with open(progress_file, 'r') as f:
                    progress = json.load(f)

                if progress.get('current_epoch', 0) < start_epoch - 1:
                    print(f"   [FIX] Patching stale JSON: {progress.get('current_epoch')} -> {start_epoch - 1}")
                    progress['current_epoch'] = start_epoch - 1
                    progress['early_stopped'] = False
                    with open(progress_file, 'w') as f:
                        json.dump(progress, f, indent=2)

        except Exception as e:
            print(f"   Error loading checkpoint: {e}")
            print(f"   Starting fresh from Epoch 1")
    else:
        # Initialize JSON if starting fresh
        if os.path.exists(progress_file):
            with open(progress_file, 'r') as f:
                progress = json.load(f)
        else:
            progress = {
                "current_fold": 4,
                "current_epoch": 0,
                "completed_folds": [1, 2, 3],
                "fold_results": [],
                "early_stopped": False
            }

    # 3. Train Loop
    for epoch in range(start_epoch, NUM_EPOCHS + 1):
        print(f"\nFinal Training - Epoch {epoch}/{NUM_EPOCHS}")

        train_loss, train_p, train_l = train_epoch(final_model, final_train_loader, criterion, optimizer, DEVICE)
        test_loss, test_p, test_l = validate(final_model, final_test_loader, criterion, DEVICE)

        train_mcc = matthews_corrcoef(train_l, (train_p > 0.5).astype(int))
        test_metrics = calculate_metrics(test_l, test_p)

        print(f"Loss: {train_loss:.4f} (Train) | {test_loss:.4f} (Test)")
        print(f"MCC:  {train_mcc:.4f} (Train) | {test_metrics['mcc']:.4f} (Test)")

        # --- UPDATE JSON LIVE ---
        progress['current_epoch'] = epoch
        with open(progress_file, 'w') as f:
            json.dump(progress, f, indent=2)
        # ------------------------

        if test_metrics['mcc'] > best_mcc:
            best_mcc = test_metrics['mcc']
            no_improve = 0
            torch.save({
                'epoch': epoch,
                'model_state_dict': final_model.state_dict(),
                'test_metrics': test_metrics,
                'architecture': 'CTRhythm'
            }, final_save_path)
            print("✓ Saved New Best Model")
        else:
            no_improve += 1
            print(f"No improvement ({no_improve}/{EARLY_STOP_PATIENCE})")

        if no_improve >= EARLY_STOP_PATIENCE:
            print("Early stopping triggered.")
            progress['early_stopped'] = True
            with open(progress_file, 'w') as f:
                json.dump(progress, f, indent=2)
            break

    print("\n" + "="*80)
    print("FINAL TEST EVALUATION")
    print("="*80)

    # Load best model safely (weights_only=False)
    if os.path.exists(final_save_path):
        checkpoint = torch.load(final_save_path, weights_only=False)
        final_model.load_state_dict(checkpoint['model_state_dict'])

    _, final_p, final_l = validate(final_model, final_test_loader, criterion, DEVICE)
    final_metrics = calculate_metrics(final_l, final_p)

    cm_final_path = os.path.join(SAVE_DIR, "confusion_matrix_final.png")
    plot_confusion_matrix(final_metrics['confusion_matrix'], "CTRhythm - Final Test Confusion Matrix", cm_final_path)

    print(f"\nFinal Test Results:")
    print(f"  Accuracy:    {final_metrics['accuracy']:.4f}")
    print(f"  Precision:   {final_metrics['precision']:.4f}")
    print(f"  Sensitivity: {final_metrics['sensitivity']:.4f}")
    print(f"  Specificity: {final_metrics['specificity']:.4f}")
    print(f"  F1-Score:    {final_metrics['f1']:.4f}")
    print(f"  MCC:         {final_metrics['mcc']:.4f}")
    print(f"  AUC-ROC:     {final_metrics['auc_roc']:.4f}")
    print(f"  AUPRC:       {final_metrics['auprc']:.4f}")

    final_results_path = os.path.join(SAVE_DIR, "final_test_results.json")
    with open(final_results_path, 'w') as f:
        serializable_mets = {k: (v.tolist() if isinstance(v, np.ndarray) else v)
                            for k, v in final_metrics.items()}
        json.dump(serializable_mets, f, indent=2)

    print("\n" + "="*80)
    print(f"All models and results saved in: {SAVE_DIR}")
    print("="*80)


if __name__ == "__main__":
    ctrhythm_3fold_cv()

# XAI: Grad-CAM on DDNN

## Generating Grad-CAM

In [ ]:
# ============================================================
# GradCAM on DDNN — Test Split Only (Both Leads)
# ============================================================

import os
import json
import h5py
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

# ============================================================
# CONFIGURATION
# ============================================================
DATA_DIR   = "/kaggle/input/datasets/sheianne/shdbiridiaprocessed"
H5_FILE    = os.path.join(DATA_DIR, "afib_preprocessed.h5")
SPLIT_FILE = os.path.join(DATA_DIR, "dataset_split.json")
MODEL_PATH = "/kaggle/input/datasets/sheianne/bestmodel/ddnn_FINAL.pth"
SAVE_DIR   = "/kaggle/working/gradcam"
TARGET_FS  = 125
N_LEADS    = 2   # H5 stores 2 leads per segment

os.makedirs(SAVE_DIR, exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

# White → Red colormap (no green)
CMAP = LinearSegmentedColormap.from_list("white_red", ["white", "red"])


# ============================================================
# MODEL DEFINITION
# ============================================================

class SEBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.squeeze    = nn.AdaptiveAvgPool1d(1)
        self.excitation = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _ = x.size()
        y = self.squeeze(x).view(b, c)
        y = self.excitation(y).view(b, c, 1)
        return x * y.expand_as(x)


class DenseBlock(nn.Module):
    def __init__(self, in_channels, growth_rate, num_layers):
        super().__init__()
        self.layers = nn.ModuleList()
        for i in range(num_layers):
            self.layers.append(nn.Sequential(
                nn.BatchNorm1d(in_channels + i * growth_rate),
                nn.ReLU(inplace=True),
                nn.Conv1d(in_channels + i * growth_rate, growth_rate,
                          kernel_size=3, padding=1, bias=False)
            ))

    def forward(self, x):
        features = [x]
        for layer in self.layers:
            features.append(layer(torch.cat(features, 1)))
        return torch.cat(features, 1)


class TransitionLayer(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.bn   = nn.BatchNorm1d(in_channels)
        self.conv = nn.Conv1d(in_channels, out_channels, kernel_size=1, bias=False)
        self.pool = nn.AvgPool1d(kernel_size=2, stride=2)

    def forward(self, x):
        return self.pool(self.conv(F.relu(self.bn(x))))


class DDNN(nn.Module):
    def __init__(self, in_channels=1, growth_rate=6, block_config=[2, 4, 6, 4],
                 reduction=0.5, num_classes=1):
        super().__init__()
        self.stem_conv1 = nn.Conv1d(in_channels, 16, kernel_size=7,  padding=3,  bias=False)
        self.stem_conv2 = nn.Conv1d(in_channels, 16, kernel_size=15, padding=7,  bias=False)
        self.stem_conv3 = nn.Conv1d(in_channels, 16, kernel_size=23, padding=11, bias=False)
        self.stem_bn    = nn.BatchNorm1d(48)
        self.stem_relu  = nn.ReLU(inplace=True)
        self.stem_pool  = nn.MaxPool1d(kernel_size=3, stride=2, padding=1)

        num_features = 48
        self.blocks      = nn.ModuleList()
        self.se_blocks   = nn.ModuleList()
        self.transitions = nn.ModuleList()

        for i, num_layers in enumerate(block_config):
            self.se_blocks.append(SEBlock(num_features, reduction=16))
            block = DenseBlock(num_features, growth_rate, num_layers)
            self.blocks.append(block)
            num_features += num_layers * growth_rate
            if i != len(block_config) - 1:
                out_features = int(num_features * reduction)
                self.transitions.append(TransitionLayer(num_features, out_features))
                num_features = out_features

        self.final_bn    = nn.BatchNorm1d(num_features)
        self.global_pool = nn.AdaptiveAvgPool1d(1)
        self.fc          = nn.Linear(num_features, num_classes)

    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = torch.cat([self.stem_conv1(x), self.stem_conv2(x), self.stem_conv3(x)], dim=1)
        x = self.stem_relu(self.stem_bn(x))
        x = self.stem_pool(x)
        for i, (se, block) in enumerate(zip(self.se_blocks, self.blocks)):
            x = se(x)
            x = block(x)
            if i < len(self.transitions):
                x = self.transitions[i](x)
        x = F.relu(self.final_bn(x))
        x = self.global_pool(x).squeeze(-1)
        return self.fc(x)


# ============================================================
# GRADCAM
# ============================================================

class GradCAM1D:
    def __init__(self, model, target_layer):
        self.model        = model
        self.target_layer = target_layer
        self.gradients    = None
        self.activations  = None
        self._register_hooks()

    def _register_hooks(self):
        def fwd(module, inp, out): self.activations = out.detach()
        def bwd(module, gin, gout): self.gradients = gout[0].detach()
        self.target_layer.register_forward_hook(fwd)
        self.target_layer.register_full_backward_hook(bwd)

    def generate(self, x):
        """
        x: (1, 3750, 1) — single lead tensor
        Returns: cam (3750,), logit (float), prob (float)
        """
        self.model.eval()
        x = x.to(DEVICE)
        logit = self.model(x)
        prob  = torch.sigmoid(logit).item()

        self.model.zero_grad()
        logit.backward()

        weights = self.gradients.mean(dim=-1, keepdim=True)
        cam     = F.relu((weights * self.activations).sum(dim=1))
        cam     = cam.squeeze(0).cpu().numpy()

        cam_up = F.interpolate(
            torch.tensor(cam).unsqueeze(0).unsqueeze(0),
            size=x.shape[1], mode='linear', align_corners=False
        ).squeeze().numpy()

        cam_min, cam_max = cam_up.min(), cam_up.max()
        cam_up = (cam_up - cam_min) / (cam_max - cam_min + 1e-8)
        return cam_up, logit.item(), prob


# ============================================================
# DATA LOADING
# ============================================================

def load_test_split_indices(h5_path, split_file):
    """
    Returns list of (global_h5_index, label) for test patients only.
    Also prints raw segment count and lead-agnostic equivalent.
    """
    with open(split_file, 'r') as f:
        split = json.load(f)
    test_patients = set(split['test_patients'])
    print(f"Test patients loaded : {len(test_patients)}")

    with h5py.File(h5_path, 'r') as f:
        all_groups = [g.decode('utf-8') if isinstance(g, bytes) else g
                      for g in f['groups'][:]]
        all_labels = f['y'][:]

    test_index_label = [
        (i, int(all_labels[i]))
        for i, g in enumerate(all_groups)
        if g in test_patients
    ]

    n_total  = len(test_index_label)
    n_afib   = sum(1 for _, lbl in test_index_label if lbl == 1)
    n_normal = n_total - n_afib

    print(f"\n=== Test Split Segment Counts ===")
    print(f"  Raw H5 windows       : {n_total:>10,}   (unique ECG segments)")
    print(f"  Lead-agnostic total  : {n_total * N_LEADS:>10,}   (× {N_LEADS} leads, as seen during training)")
    print(f"  — AFib  windows      : {n_afib:>10,}   ({n_afib * N_LEADS:,} lead-agnostic)")
    print(f"  — Normal windows     : {n_normal:>10,}   ({n_normal * N_LEADS:,} lead-agnostic)")
    print(f"=================================\n")

    return test_index_label


def find_afib_in_test(test_index_label):
    """Return the first H5 index in the test split with label=1 (AFib)."""
    for idx, lbl in test_index_label:
        if lbl == 1:
            return idx
    raise RuntimeError("No AFib segment found in the test split.")


def find_normal_in_test(test_index_label):
    """Return the first H5 index in the test split with label=0 (Normal)."""
    for idx, lbl in test_index_label:
        if lbl == 0:
            return idx
    raise RuntimeError("No Normal segment found in the test split.")


def load_segment(h5_path, global_idx):
    """
    Load a segment from the H5 file.
    Returns X (3750, 2), true_label, group string.
    """
    with h5py.File(h5_path, 'r') as f:
        X     = f['X'][global_idx].astype(np.float32)   # (3750, 2)
        y     = int(f['y'][global_idx])
        group = f['groups'][global_idx]
        if isinstance(group, bytes):
            group = group.decode('utf-8')
    print(f"Segment #{global_idx} | Patient: {group} | "
          f"Label: {'AFib' if y == 1 else 'Normal'}")
    return X, y, group


# ============================================================
# PLOTTING
# ============================================================

def plot_gradcam(ecg_signal, cam, true_label, prob, group,
                 segment_idx, lead_num, fs=125, save_path=None):
    """
    ecg_signal : 1-D array (3750,) for the chosen lead
    cam        : 1-D GradCAM array (3750,), normalised [0, 1]
    lead_num   : 0 or 1 (for display only)
    """
    T          = len(ecg_signal)
    time_s     = np.arange(T) / fs
    pred_label = 1 if prob >= 0.5 else 0
    correct    = pred_label == true_label

    fig, axes = plt.subplots(2, 1, figsize=(16, 7),
                             gridspec_kw={'height_ratios': [3, 1]})
    fig.suptitle(
        f"GradCAM — DDNN  |  Segment #{segment_idx}  |  Patient: {group}  |  Lead {lead_num}\n"
        f"True: {'AFib' if true_label == 1 else 'Normal'}   "
        f"Pred: {'AFib' if pred_label == 1 else 'Normal'}   "
        f"P(AFib) = {prob:.3f}   "
        f"({'✓ Correct' if correct else '✗ Incorrect'})",
        fontsize=13, fontweight='bold'
    )

    # ── Top: ECG + white-to-red heatmap overlay ────────────────
    ax0 = axes[0]
    ax0.plot(time_s, ecg_signal, color='#1a1a2e', linewidth=0.8, zorder=3)
    ax0.set_ylabel("Normalised Amplitude", fontsize=11)

    for t in range(T - 1):
        intensity = float(cam[t])
        ax0.axvspan(time_s[t], time_s[t + 1],
                    ymin=0, ymax=1,
                    alpha=0.6 * intensity + 0.02,
                    color=CMAP(intensity), zorder=1)

    sm = plt.cm.ScalarMappable(cmap=CMAP, norm=plt.Normalize(0, 1))
    sm.set_array([])
    fig.colorbar(sm, ax=ax0, orientation='vertical',
                 fraction=0.015, pad=0.01).set_label("GradCAM Importance", fontsize=9)
    ax0.set_xlim(time_s[0], time_s[-1])
    ax0.grid(axis='x', linestyle='--', alpha=0.4)

    # ── Bottom: CAM activation curve ───────────────────────────
    ax1 = axes[1]
    ax1.fill_between(time_s, cam, alpha=0.7, color='red', label='GradCAM')
    ax1.plot(time_s, cam, color='darkred', linewidth=0.8)
    ax1.axhline(0.5, linestyle='--', color='gray', linewidth=0.8, label='0.5 threshold')
    ax1.set_xlabel("Time (s)", fontsize=11)
    ax1.set_ylabel("Activation", fontsize=11)
    ax1.set_xlim(time_s[0], time_s[-1])
    ax1.set_ylim(0, 1.05)
    ax1.legend(fontsize=9, loc='upper right')
    ax1.grid(axis='x', linestyle='--', alpha=0.4)

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    if save_path:
        plt.savefig(save_path, dpi=200, bbox_inches='tight')
        print(f"  Saved → {save_path}")
    plt.show()
    plt.close()


# ============================================================
# MAIN
# ============================================================

def run_gradcam_test_split():
    # ── 1. Load model ──────────────────────────────────────────
    print("Loading DDNN model...")
    model = DDNN(in_channels=1, growth_rate=6, block_config=[2, 4, 6, 4],
                 reduction=0.5, num_classes=1).to(DEVICE)
    checkpoint = torch.load(MODEL_PATH, map_location=DEVICE, weights_only=False)
    model.load_state_dict(checkpoint['model'])
    model.eval()
    print("Model loaded.\n")

    target_layer = model.blocks[-1].layers[-1][-1]
    gradcam      = GradCAM1D(model, target_layer)

    # ── 2. Build test-split index map ──────────────────────────
    test_index_label = load_test_split_indices(H5_FILE, SPLIT_FILE)

    # ── 3. Run GradCAM: one AFib + one Normal, both leads each ─
    candidates = {
        "afib":   find_afib_in_test(test_index_label),
        "normal": find_normal_in_test(test_index_label),
    }

    for label_name, segment_idx in candidates.items():
        print(f"\n{'='*60}")
        print(f"  Segment type : {label_name.upper()}  (H5 index #{segment_idx})")
        print(f"{'='*60}")

        X, true_label, group = load_segment(H5_FILE, segment_idx)

        # Run GradCAM independently for each lead
        for lead_num in range(N_LEADS):
            print(f"\n  → Lead {lead_num}")

            X_lead   = X[:, lead_num:lead_num+1]                     # (3750, 1)
            x_tensor = torch.FloatTensor(X_lead).unsqueeze(0)        # (1, 3750, 1)

            cam, logit, prob = gradcam.generate(x_tensor)
            pred_label       = 1 if prob >= 0.5 else 0

            print(f"    Logit      : {logit:.4f}")
            print(f"    P(AFib)    : {prob:.4f}")
            print(f"    Prediction : {'AFib' if pred_label == 1 else 'Normal'}")
            print(f"    True Label : {'AFib' if true_label == 1 else 'Normal'}")
            print(f"    Correct?   : {'✓ YES' if pred_label == true_label else '✗ NO'}")

            save_path = os.path.join(
                SAVE_DIR,
                f"gradcam_test_{label_name}_seg{segment_idx}_lead{lead_num}.png"
            )
            plot_gradcam(X_lead[:, 0], cam, true_label, prob, group,
                         segment_idx, lead_num, fs=TARGET_FS, save_path=save_path)

            np.save(save_path.replace('.png', '.npy'), cam)


if __name__ == "__main__":
    run_gradcam_test_split()

## Perturbation Analysis (Simic et al.)

### AUPC (Area Under the Perturbation Curve)

In [ ]:
# ============================================================
# AUPC — Area Under the Perturbation Curve
# Assumes: DDNN, GradCAM1D, load_segment, H5_FILE, DEVICE
#          are already defined in the previous cell.
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import torch
from matplotlib.colors import LinearSegmentedColormap

CMAP = LinearSegmentedColormap.from_list("white_red", ["white", "red"])

AUPC_SAVE_DIR = "/kaggle/working/gradcam/aupc"
N_STEPS       = 20    # perturbation steps (5% of signal each)
TARGET_FS     = 125
os.makedirs(AUPC_SAVE_DIR, exist_ok=True)


# ============================================================
# CORE AUPC COMPUTATION
# ============================================================

def _get_confidence(model, X_np):
    """Run model on a single (3750, 1) numpy array and return P(AFib)."""
    x = torch.FloatTensor(X_np).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        logit = model(x)
    return torch.sigmoid(logit).item()


def compute_aupc(model, gradcam, X, n_steps=N_STEPS, perturb_value=0.0):
    """
    Compute MoRF and LeRF perturbation curves and their AUPCs.

    Parameters
    ----------
    model         : trained DDNN (eval mode)
    gradcam       : GradCAM1D instance
    X             : np.ndarray of shape (3750, 1) — single lead, z-normalised
    n_steps       : number of perturbation steps (fraction = 1/n_steps each)
    perturb_value : replacement value for masked timesteps (0.0 = z-score mean)

    Returns
    -------
    results : dict with keys:
        'fractions'   — x-axis, shape (n_steps+1,)
        'morf_curve'  — P(AFib) at each step, MoRF order
        'lerf_curve'  — P(AFib) at each step, LeRF order
        'morf_aupc'   — scalar AUC of MoRF curve
        'lerf_aupc'   — scalar AUC of LeRF curve
        'cam'         — raw GradCAM saliency (3750,)
    """
    T         = X.shape[0]
    step_size = T // n_steps
    fractions = np.linspace(0, 1, n_steps + 1)

    # ── Step 1: Get GradCAM saliency ─────────────────────────
    x_tensor     = torch.FloatTensor(X).unsqueeze(0)
    cam, _, base_prob = gradcam.generate(x_tensor)

    # ── Step 2: Rank timesteps ────────────────────────────────
    # sorted_morf[0] = most important timestep index
    sorted_morf = np.argsort(cam)[::-1].copy()
    sorted_lerf = np.argsort(cam).copy()          # least important first

    # ── Step 3: Build perturbation curves ────────────────────
    def perturbation_curve(sorted_indices):
        curve = [base_prob]   # step 0: no masking
        for step in range(1, n_steps + 1):
            mask_idx        = sorted_indices[: step * step_size]
            X_perturbed     = X.copy()
            X_perturbed[mask_idx, :] = perturb_value
            curve.append(_get_confidence(model, X_perturbed))
        return np.array(curve)

    morf_curve = perturbation_curve(sorted_morf)
    lerf_curve = perturbation_curve(sorted_lerf)

    # ── Step 4: AUPC via trapezoidal rule ─────────────────────
    morf_aupc = float(np.trapezoid(morf_curve, fractions))
    lerf_aupc = float(np.trapezoid(lerf_curve, fractions))

    return {
        'fractions':  fractions,
        'morf_curve': morf_curve,
        'lerf_curve': lerf_curve,
        'morf_aupc':  morf_aupc,
        'lerf_aupc':  lerf_aupc,
        'cam':        cam,
    }


# ============================================================
# PLOTTING
# ============================================================

def plot_aupc(results, ecg_signal, true_label, prob, group,
              segment_idx, lead_num, fs=TARGET_FS, save_path=None):
    """
    3-panel figure:
      Top-left  — ECG with GradCAM heatmap
      Top-right — GradCAM activation curve
      Bottom    — MoRF & LeRF perturbation curves with AUPC shading
    """
    fractions  = results['fractions']
    morf_curve = results['morf_curve']
    lerf_curve = results['lerf_curve']
    morf_aupc  = results['morf_aupc']
    lerf_aupc  = results['lerf_aupc']
    cam        = results['cam']

    T      = len(ecg_signal)
    time_s = np.arange(T) / fs
    pred_label = 1 if prob >= 0.5 else 0

    fig = plt.figure(figsize=(18, 10))
    gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.4, wspace=0.35)

    ax_ecg  = fig.add_subplot(gs[0, 0])
    ax_cam  = fig.add_subplot(gs[0, 1])
    ax_aupc = fig.add_subplot(gs[1, :])    # full-width bottom

    fig.suptitle(
        f"AUPC — DDNN  |  Segment #{segment_idx}  |  Patient: {group}  |  Lead {lead_num}\n"
        f"True: {'AFib' if true_label == 1 else 'Normal'}   "
        f"Pred: {'AFib' if pred_label == 1 else 'Normal'}   "
        f"P(AFib) = {prob:.3f}",
        fontsize=13, fontweight='bold'
    )

    # ── Panel 1: ECG + heatmap ───────────────────────────────
    ax_ecg.plot(time_s, ecg_signal, color='#1a1a2e', linewidth=0.7, zorder=3)
    for t in range(T - 1):
        intensity = float(cam[t])
        ax_ecg.axvspan(time_s[t], time_s[t+1],
                       ymin=0, ymax=1,
                       alpha=0.6 * intensity + 0.02,
                       color=CMAP(intensity), zorder=1)
    sm = plt.cm.ScalarMappable(cmap=CMAP, norm=plt.Normalize(0, 1))
    sm.set_array([])
    fig.colorbar(sm, ax=ax_ecg, fraction=0.04, pad=0.02).set_label("GradCAM", fontsize=8)
    ax_ecg.set_xlabel("Time (s)", fontsize=10)
    ax_ecg.set_ylabel("Amplitude", fontsize=10)
    ax_ecg.set_title("ECG + GradCAM Heatmap", fontsize=11)
    ax_ecg.set_xlim(time_s[0], time_s[-1])
    ax_ecg.grid(axis='x', linestyle='--', alpha=0.3)

    # ── Panel 2: GradCAM activation ─────────────────────────
    ax_cam.fill_between(time_s, cam, alpha=0.6, color='red')
    ax_cam.plot(time_s, cam, color='darkred', linewidth=0.7)
    ax_cam.axhline(0.5, linestyle='--', color='gray', linewidth=0.8)
    ax_cam.set_xlabel("Time (s)", fontsize=10)
    ax_cam.set_ylabel("Activation", fontsize=10)
    ax_cam.set_title("GradCAM Activation Curve", fontsize=11)
    ax_cam.set_xlim(time_s[0], time_s[-1])
    ax_cam.set_ylim(0, 1.05)
    ax_cam.grid(axis='x', linestyle='--', alpha=0.3)

    # ── Panel 3: Perturbation curves + AUPC shading ──────────
    pct = fractions * 100   # convert to percentage for x-axis

    # MoRF
    ax_aupc.plot(pct, morf_curve, 'o-', color='crimson', linewidth=2,
                 markersize=5, label=f'MoRF  (AUPC = {morf_aupc:.4f})')
    ax_aupc.fill_between(pct, morf_curve, alpha=0.15, color='crimson')

    # LeRF
    ax_aupc.plot(pct, lerf_curve, 's--', color='steelblue', linewidth=2,
                 markersize=5, label=f'LeRF  (AUPC = {lerf_aupc:.4f})')
    ax_aupc.fill_between(pct, lerf_curve, alpha=0.15, color='steelblue')

    # Baseline (random) — expected confidence if masking were uninformative
    ax_aupc.axhline(0.5, linestyle=':', color='gray', linewidth=1, label='Decision boundary (0.5)')

    ax_aupc.set_xlabel("Features Removed (%)", fontsize=11)
    ax_aupc.set_ylabel("P(AFib)", fontsize=11)
    ax_aupc.set_title(
        f"Perturbation Curve — MoRF vs LeRF  |  {N_STEPS} steps ({100//N_STEPS}% each)\n"
        f"Lower MoRF AUPC → more faithful explanation   |   "
        f"Higher LeRF AUPC → more faithful explanation",
        fontsize=10
    )
    ax_aupc.set_xlim(0, 100)
    ax_aupc.set_ylim(-0.05, 1.05)
    ax_aupc.legend(fontsize=10, loc='upper right')
    ax_aupc.grid(linestyle='--', alpha=0.4)

    if save_path:
        plt.savefig(save_path, dpi=200, bbox_inches='tight')
        print(f"  Saved → {save_path}")
    plt.show()
    plt.close()


# ============================================================
# MAIN
# ============================================================

def run_aupc_test_split():
    # ── Reload model (new cell, same checkpoint) ──────────────
    print("Loading DDNN model...")
    model = DDNN(in_channels=1, growth_rate=6, block_config=[2, 4, 6, 4],
                 reduction=0.5, num_classes=1).to(DEVICE)
    checkpoint = torch.load(MODEL_PATH, map_location=DEVICE, weights_only=False)
    model.load_state_dict(checkpoint['model'])
    model.eval()
    print("Model loaded.\n")

    target_layer = model.blocks[-1].layers[-1][-1]
    gradcam      = GradCAM1D(model, target_layer)

    # ── Target segments (from previous cell output) ────────────
    # AFib:   H5 index #13815  (SHDB_131)
    # Normal: H5 index #9210   (SHDB_122)
    targets = [
        {"label_name": "afib",   "segment_idx": 13815},
        {"label_name": "normal", "segment_idx": 9210},
    ]

    summary = []

    for target in targets:
        segment_idx = target["segment_idx"]
        label_name  = target["label_name"]

        X, true_label, group = load_segment(H5_FILE, segment_idx)

        for lead_num in range(N_LEADS):
            print(f"\n{'='*60}")
            print(f"  AUPC: {label_name.upper()}  |  Segment #{segment_idx}  |  Lead {lead_num}")
            print(f"{'='*60}")

            X_lead   = X[:, lead_num:lead_num+1]          # (3750, 1)
            x_tensor = torch.FloatTensor(X_lead).unsqueeze(0)

            # Get baseline P(AFib) for title
            with torch.no_grad():
                logit = model(x_tensor.to(DEVICE))
            base_prob = torch.sigmoid(logit).item()

            # Compute AUPC
            results = compute_aupc(model, gradcam, X_lead, n_steps=N_STEPS)

            print(f"  MoRF AUPC : {results['morf_aupc']:.4f}  "
                  f"(lower = more faithful)")
            print(f"  LeRF AUPC : {results['lerf_aupc']:.4f}  "
                  f"(higher = more faithful)")

            save_path = os.path.join(
                AUPC_SAVE_DIR,
                f"aupc_{label_name}_seg{segment_idx}_lead{lead_num}.png"
            )
            plot_aupc(
                results     = results,
                ecg_signal  = X_lead[:, 0],
                true_label  = true_label,
                prob        = base_prob,
                group       = group,
                segment_idx = segment_idx,
                lead_num    = lead_num,
                save_path   = save_path,
            )

            # Save raw curve arrays
            np.save(save_path.replace('.png', '_morf.npy'), results['morf_curve'])
            np.save(save_path.replace('.png', '_lerf.npy'), results['lerf_curve'])

            summary.append({
                'label':       label_name,
                'segment_idx': segment_idx,
                'lead':        lead_num,
                'group':       group,
                'morf_aupc':   results['morf_aupc'],
                'lerf_aupc':   results['lerf_aupc'],
            })

    # ── Summary table ─────────────────────────────────────────
    print(f"\n{'='*65}")
    print(f"  {'Segment':<10} {'Patient':<14} {'Lead':<6} {'Label':<8} {'MoRF AUPC':<12} {'LeRF AUPC'}")
    print(f"  {'-'*63}")
    for r in summary:
        print(f"  #{r['segment_idx']:<9} {r['group']:<14} {r['lead']:<6} "
              f"{r['label']:<8} {r['morf_aupc']:<12.4f} {r['lerf_aupc']:.4f}")
    print(f"{'='*65}")


if __name__ == "__main__":
    run_aupc_test_split()

### DDS (Decaying Degradation Score)

In [ ]:
# ============================================================
# DDS — Decaying Degradation Score
# Assumes: DDNN, GradCAM1D, compute_aupc, load_segment,
#          H5_FILE, DEVICE, CMAP, N_LEADS defined previously.
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import torch
import os

DDS_SAVE_DIR = "/kaggle/working/gradcam/dds"
N_STEPS      = 20
TARGET_FS    = 125
os.makedirs(DDS_SAVE_DIR, exist_ok=True)


# ============================================================
# CORE DDS COMPUTATION
# ============================================================

def compute_dds(morf_curve, lerf_curve):
    """
    Decaying Degradation Score (DDS).

    Formula:
        DDS     = Σ(i=1..n)  (L_i - M_i) · ((n - i + 1) / n)³
        DDS_max = Σ(i=1..n)  ((n - i + 1) / n)³
        DDS_norm = DDS / DDS_max  ∈ [-1, 1]

    Parameters
    ----------
    morf_curve : np.ndarray, shape (n_steps+1,)
        P(AFib) at each MoRF perturbation step. Index 0 = baseline.
    lerf_curve : np.ndarray, shape (n_steps+1,)
        P(AFib) at each LeRF perturbation step. Index 0 = baseline.

    Returns
    -------
    dds      : float  — raw DDS
    dds_max  : float  — maximum achievable DDS (perfect ranker)
    dds_norm : float  — normalised DDS in [-1, 1]
    weights  : np.ndarray — cubic decay weights for each step
    """
    n      = len(morf_curve) - 1          # number of perturbation steps
    L      = lerf_curve[1:]               # skip baseline; shape (n,)
    M      = morf_curve[1:]               # skip baseline; shape (n,)
    i_vals = np.arange(1, n + 1)          # i = 1, 2, ..., n

    weights  = ((n - i_vals + 1) / n) ** 3   # cubic decay from 1 → (1/n)³
    dds      = float(np.sum((L - M) * weights))
    dds_max  = float(np.sum(weights))         # achieved when L_i=1, M_i=0 ∀i
    dds_norm = dds / dds_max

    return dds, dds_max, dds_norm, weights


# ============================================================
# PLOTTING
# ============================================================

def plot_dds(results, ecg_signal, true_label, prob, group,
             segment_idx, lead_num, fs=TARGET_FS, save_path=None):
    """
    3-panel figure:
      Top-left  — ECG + GradCAM heatmap
      Top-right — Perturbation curves (MoRF & LeRF) with (L_i - M_i) gap shading
      Bottom    — Per-step weighted contribution (L_i - M_i) × weight_i
    """
    fractions  = results['fractions']
    morf_curve = results['morf_curve']
    lerf_curve = results['lerf_curve']
    cam        = results['cam']
    dds        = results['dds']
    dds_max    = results['dds_max']
    dds_norm   = results['dds_norm']
    weights    = results['weights']

    n      = len(morf_curve) - 1
    T      = len(ecg_signal)
    time_s = np.arange(T) / fs
    pred_label = 1 if prob >= 0.5 else 0

    step_labels  = np.arange(1, n + 1)
    contributions = (lerf_curve[1:] - morf_curve[1:]) * weights  # per-step contribution

    # Colour contributions: positive = correct separation (blue), negative = inverted (red)
    bar_colors = ['steelblue' if c >= 0 else 'crimson' for c in contributions]

    fig = plt.figure(figsize=(18, 10))
    gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.42, wspace=0.35)

    ax_ecg   = fig.add_subplot(gs[0, 0])
    ax_perturb = fig.add_subplot(gs[0, 1])
    ax_contrib = fig.add_subplot(gs[1, :])

    fig.suptitle(
        f"DDS — DDNN  |  Segment #{segment_idx}  |  Patient: {group}  |  Lead {lead_num}\n"
        f"True: {'AFib' if true_label == 1 else 'Normal'}   "
        f"Pred: {'AFib' if pred_label == 1 else 'Normal'}   "
        f"P(AFib) = {prob:.3f}   |   "
        f"DDS = {dds:.4f}   DDS_max = {dds_max:.4f}   "
        f"DDS_norm = {dds_norm:.4f}",
        fontsize=12, fontweight='bold'
    )

    # ── Panel 1: ECG + heatmap ───────────────────────────────
    ax_ecg.plot(time_s, ecg_signal, color='#1a1a2e', linewidth=0.7, zorder=3)
    for t in range(T - 1):
        intensity = float(cam[t])
        ax_ecg.axvspan(time_s[t], time_s[t+1],
                       ymin=0, ymax=1,
                       alpha=0.6 * intensity + 0.02,
                       color=CMAP(intensity), zorder=1)
    sm = plt.cm.ScalarMappable(cmap=CMAP, norm=plt.Normalize(0, 1))
    sm.set_array([])
    fig.colorbar(sm, ax=ax_ecg, fraction=0.04, pad=0.02).set_label("GradCAM", fontsize=8)
    ax_ecg.set_xlabel("Time (s)", fontsize=10)
    ax_ecg.set_ylabel("Amplitude", fontsize=10)
    ax_ecg.set_title("ECG + GradCAM Heatmap", fontsize=11)
    ax_ecg.set_xlim(time_s[0], time_s[-1])
    ax_ecg.grid(axis='x', linestyle='--', alpha=0.3)

    # ── Panel 2: Perturbation curves + L-M gap ───────────────
    pct = fractions * 100
    ax_perturb.plot(pct, morf_curve, 'o-', color='crimson', linewidth=2,
                    markersize=4, label=f'MoRF (AUPC={np.trapezoid(morf_curve, fractions):.4f})')
    ax_perturb.plot(pct, lerf_curve, 's--', color='steelblue', linewidth=2,
                    markersize=4, label=f'LeRF (AUPC={np.trapezoid(lerf_curve, fractions):.4f})')

    # Shade the gap between LeRF and MoRF
    ax_perturb.fill_between(pct, morf_curve, lerf_curve,
                             where=(lerf_curve >= morf_curve),
                             alpha=0.2, color='steelblue', label='L > M (correct separation)')
    ax_perturb.fill_between(pct, morf_curve, lerf_curve,
                             where=(lerf_curve < morf_curve),
                             alpha=0.2, color='crimson', label='L < M (inverted)')

    ax_perturb.axhline(0.5, linestyle=':', color='gray', linewidth=1)
    ax_perturb.set_xlabel("Features Removed (%)", fontsize=10)
    ax_perturb.set_ylabel("P(AFib)", fontsize=10)
    ax_perturb.set_title("MoRF vs LeRF Perturbation Curves", fontsize=11)
    ax_perturb.set_xlim(0, 100)
    ax_perturb.set_ylim(-0.05, 1.05)
    ax_perturb.legend(fontsize=8, loc='upper right')
    ax_perturb.grid(linestyle='--', alpha=0.3)

    # ── Panel 3: Per-step weighted contributions ─────────────
    ax_contrib.bar(step_labels, contributions, color=bar_colors, alpha=0.8, edgecolor='black', linewidth=0.5)
    ax_contrib.axhline(0, color='black', linewidth=0.8)

    # Annotate bars with contribution values
    for i, (val, w) in enumerate(zip(contributions, weights)):
        ax_contrib.text(step_labels[i], val + (0.003 if val >= 0 else -0.008),
                        f'{val:.3f}', ha='center', va='bottom' if val >= 0 else 'top',
                        fontsize=7, color='black')

    ax_contrib.set_xlabel("Perturbation Step (i)", fontsize=10)
    ax_contrib.set_ylabel("(L_i − M_i) × weight_i", fontsize=10)
    ax_contrib.set_title(
        f"Per-Step Weighted Contribution to DDS\n"
        f"Blue = correct separation (L > M)  |  Red = inverted (M > L)  |  "
        f"DDS_norm = {dds_norm:+.4f}",
        fontsize=10
    )
    ax_contrib.set_xticks(step_labels)
    ax_contrib.grid(axis='y', linestyle='--', alpha=0.4)

    plt.tight_layout(rect=[0, 0, 1, 0.93])
    if save_path:
        plt.savefig(save_path, dpi=200, bbox_inches='tight')
        print(f"  Saved → {save_path}")
    plt.show()
    plt.close()


# ============================================================
# MAIN
# ============================================================

def run_dds_test_split():
    # ── Reload model ──────────────────────────────────────────
    print("Loading DDNN model...")
    model = DDNN(in_channels=1, growth_rate=6, block_config=[2, 4, 6, 4],
                 reduction=0.5, num_classes=1).to(DEVICE)
    checkpoint = torch.load(MODEL_PATH, map_location=DEVICE, weights_only=False)
    model.load_state_dict(checkpoint['model'])
    model.eval()
    print("Model loaded.\n")

    target_layer = model.blocks[-1].layers[-1][-1]
    gradcam      = GradCAM1D(model, target_layer)

    # ── Same segments as AUPC ─────────────────────────────────
    targets = [
        {"label_name": "afib",   "segment_idx": 13815},
        {"label_name": "normal", "segment_idx": 9210},
    ]

    summary = []

    for target in targets:
        segment_idx = target["segment_idx"]
        label_name  = target["label_name"]

        X, true_label, group = load_segment(H5_FILE, segment_idx)

        for lead_num in range(N_LEADS):
            print(f"\n{'='*60}")
            print(f"  DDS: {label_name.upper()}  |  Segment #{segment_idx}  |  Lead {lead_num}")
            print(f"{'='*60}")

            X_lead   = X[:, lead_num:lead_num+1]
            x_tensor = torch.FloatTensor(X_lead).unsqueeze(0)

            # Baseline P(AFib)
            with torch.no_grad():
                prob = torch.sigmoid(model(x_tensor.to(DEVICE))).item()

            # Perturbation curves (reuses AUPC logic)
            aupc_results = compute_aupc(model, gradcam, X_lead, n_steps=N_STEPS)

            # DDS
            dds, dds_max, dds_norm, weights = compute_dds(
                aupc_results['morf_curve'],
                aupc_results['lerf_curve']
            )

            print(f"  DDS (raw)    : {dds:.4f}")
            print(f"  DDS_max      : {dds_max:.4f}")
            print(f"  DDS_norm     : {dds_norm:+.4f}  ∈ [-1, 1]")
            print(f"  Interpretation: ", end="")
            if dds_norm >= 0.5:
                print("Highly faithful — GradCAM cleanly separates relevant from irrelevant")
            elif dds_norm >= 0.2:
                print("Moderately faithful")
            elif dds_norm >= 0.0:
                print("Weakly faithful — minimal separation")
            else:
                print("Inverted — GradCAM reverses feature relevance")

            # Combine results for plotting
            plot_results = {
                **aupc_results,
                'dds': dds, 'dds_max': dds_max, 'dds_norm': dds_norm, 'weights': weights
            }

            save_path = os.path.join(
                DDS_SAVE_DIR,
                f"dds_{label_name}_seg{segment_idx}_lead{lead_num}.png"
            )
            plot_dds(
                results     = plot_results,
                ecg_signal  = X_lead[:, 0],
                true_label  = true_label,
                prob        = prob,
                group       = group,
                segment_idx = segment_idx,
                lead_num    = lead_num,
                save_path   = save_path,
            )

            summary.append({
                'label':       label_name,
                'segment_idx': segment_idx,
                'lead':        lead_num,
                'group':       group,
                'dds':         dds,
                'dds_max':     dds_max,
                'dds_norm':    dds_norm,
            })

    # ── Summary table ─────────────────────────────────────────
    print(f"\n{'='*70}")
    print(f"  {'Segment':<10} {'Patient':<14} {'Lead':<6} {'Label':<8} "
          f"{'DDS':>8} {'DDS_max':>8} {'DDS_norm':>10}")
    print(f"  {'-'*68}")
    for r in summary:
        print(f"  #{r['segment_idx']:<9} {r['group']:<14} {r['lead']:<6} "
              f"{r['label']:<8} {r['dds']:>8.4f} {r['dds_max']:>8.4f} {r['dds_norm']:>+10.4f}")
    print(f"{'='*70}")


if __name__ == "__main__":
    run_dds_test_split()

### PES (Perturbation Effect Size)

In [ ]:
# ============================================================
# PES — Perturbation Effect Size (Full Test Split)
# With Kaggle-compatible checkpoint/resume logic
# ============================================================

import os
import json
import h5py
import shutil
import numpy as np
import torch
import torch.nn.functional as F
from tqdm import tqdm
import time

PES_SAVE_DIR = os.path.join(SAVE_DIR, "pes")
N_STEPS      = 20
CHECKPOINT_EVERY = 5000

# Where you upload your checkpoint .npz between Kaggle sessions
# (upload pes_checkpoint.npz as a dataset, then point here)
INPUT_CHECKPOINT_PATH = "/kaggle/input/datasets/sheianne/pes-checkpoint"

CHECKPOINT_FILE = os.path.join(PES_SAVE_DIR, "pes_checkpoint.npz")
RESULTS_FILE    = os.path.join(PES_SAVE_DIR, "pes_results.npz")


# ============================================================
# RESUME ENVIRONMENT (same pattern as training)
# ============================================================

def setup_resume_environment(input_root, save_dir):
    """
    Copy checkpoint files from read-only input dataset to
    writable save directory at the start of each session.
    Copies: .npz, .json, .pth
    """
    print(f"\nSetting up resume environment...")
    print(f"   Input checkpoint path : {input_root}")
    print(f"   Save directory        : {save_dir}")

    os.makedirs(save_dir, exist_ok=True)
    print(f"   Save directory created/verified")

    if not os.path.exists(input_root):
        print(f"   No input checkpoint dataset found — starting fresh")
        return

    copied_files = 0
    for filename in os.listdir(input_root):
        if filename.endswith(('.npz', '.json', '.pth')):
            src_path = os.path.join(input_root, filename)
            dst_path = os.path.join(save_dir, filename)
            if not os.path.exists(dst_path):
                shutil.copy2(src_path, dst_path)
                print(f"   Copied: {filename}")
                copied_files += 1

    if copied_files == 0:
        print(f"   No checkpoint files found to copy — starting fresh")
    else:
        print(f"   Successfully copied {copied_files} file(s)")


# ============================================================
# CORE: BATCHED DDS
# ============================================================

def compute_dds_batched(model, gradcam, X, n_steps=N_STEPS):
    """
    Compute DDS_norm for a single segment-lead pair using
    batched perturbation (2×n_steps inputs in one forward pass).

    X : (3750, 1) numpy float32
    Returns: dds_norm (float) ∈ [-1, 1]
    """
    T         = X.shape[0]
    step_size = T // n_steps

    # ── GradCAM (forward + backward) ─────────────────────────
    x_t       = torch.FloatTensor(X).unsqueeze(0).to(DEVICE)
    logit     = model(x_t)
    prob_base = torch.sigmoid(logit).item()
    model.zero_grad()
    logit.backward()

    # ── Compute CAM before batch pass overwrites hooks ────────
    wgc    = gradcam.gradients.mean(dim=-1, keepdim=True)
    cam_ft = F.relu((wgc * gradcam.activations).sum(dim=1)).squeeze(0).cpu().numpy()
    cam_up = F.interpolate(
        torch.tensor(cam_ft).unsqueeze(0).unsqueeze(0),
        size=T, mode='linear', align_corners=False
    ).squeeze().numpy()
    c_min, c_max = cam_up.min(), cam_up.max()
    cam_up = (cam_up - c_min) / (c_max - c_min + 1e-8)

    # ── Rank timesteps ────────────────────────────────────────
    sorted_morf = np.argsort(cam_up)[::-1].copy()
    sorted_lerf = np.argsort(cam_up).copy()

    # ── Build batch: rows 0..n-1 = MoRF, rows n..2n-1 = LeRF ─
    batch = np.tile(X, (2 * n_steps, 1, 1))
    for step in range(1, n_steps + 1):
        m_idx = sorted_morf[: step * step_size]
        l_idx = sorted_lerf[: step * step_size]
        batch[step - 1,           m_idx, :] = 0.0
        batch[n_steps + step - 1, l_idx, :] = 0.0

    # ── Single batched forward pass ───────────────────────────
    batch_t = torch.FloatTensor(batch).to(DEVICE)
    with torch.no_grad():
        probs_batch = torch.sigmoid(
            model(batch_t)
        ).squeeze(-1).cpu().numpy()

    morf_curve = np.concatenate([[prob_base], probs_batch[:n_steps]])
    lerf_curve = np.concatenate([[prob_base], probs_batch[n_steps:]])

    # ── DDS formula ───────────────────────────────────────────
    i_vals  = np.arange(1, n_steps + 1)
    weights = ((n_steps - i_vals + 1) / n_steps) ** 3
    dds     = float(np.sum((lerf_curve[1:] - morf_curve[1:]) * weights))
    dds_max = float(np.sum(weights))
    return dds / dds_max


# ============================================================
# PES FORMULA
# ============================================================

def compute_pes(dds_norms):
    n   = len(dds_norms)
    f   = np.sum(dds_norms > 0) / n
    u   = np.sum(dds_norms < 0) / n
    return float(f - u), float(f), float(u)


# ============================================================
# PES SUMMARY
# ============================================================

def print_pes_summary(results_arr):
    """
    results_arr columns:
      0: seg_idx   (int)
      1: lead_num  (int, 0 or 1)
      2: source    (str, 'SHDB' or 'IRIDIA')
      3: true_label (int, 0 or 1)
      4: dds_norm  (float)
    """
    dds_all  = results_arr[:, 4].astype(float)
    src_all  = results_arr[:, 2].astype(str)
    lead_all = results_arr[:, 1].astype(int)
    lbl_all  = results_arr[:, 3].astype(int)  # Extracted true_label for AFib vs Non-AFib split

    base_groups = {
        "SHDB   Lead 0" : (src_all == "SHDB")  & (lead_all == 0),
        "SHDB   Lead 1" : (src_all == "SHDB")  & (lead_all == 1),
        "IRIDIA Lead 0" : (src_all == "IRIDIA") & (lead_all == 0),
        "IRIDIA Lead 1" : (src_all == "IRIDIA") & (lead_all == 1),
        "SHDB   Overall": (src_all == "SHDB"),
        "IRIDIA Overall": (src_all == "IRIDIA"),
        "Overall"       : np.ones(len(dds_all), dtype=bool),
    }

    print(f"\n{'='*76}")
    print(f"  PERTURBATION EFFECT SIZE (PES) — GradCAM on DDNN")
    print(f"  Evaluated : {len(dds_all):,} segment-lead pairs")
    print(f"{'='*76}")
    print(f"  {'Group':<30} {'N':>8} {'f (DDS>0)':>11} {'u (DDS<0)':>11} {'PES':>8}")
    print(f"  {'-'*74}")
    
    for gname, mask in base_groups.items():
        # Iterate over All, AFib, Non-AFib for each group
        for sub_name, sub_mask in [
            (f"{gname} (All)", mask),
            (f"{gname} (AFib)", mask & (lbl_all == 1)),
            (f"{gname} (Non-AF)", mask & (lbl_all == 0))
        ]:
            subset = dds_all[sub_mask]
            if len(subset) == 0:
                print(f"  {sub_name:<30} {'—':>8}")
                continue
            pes, f, u = compute_pes(subset)
            print(f"  {sub_name:<30} {len(subset):>8,} {f*100:>10.2f}% {u*100:>10.2f}% {pes:>+8.4f}")
        print(f"  {'-'*74}")

    print(f"{'='*76}")

    print(f"\n  DDS_norm distribution (overall):")
    print(f"    Mean   : {np.mean(dds_all):+.4f}")
    print(f"    Median : {np.median(dds_all):+.4f}")
    print(f"    Std    : {np.std(dds_all):.4f}")
    print(f"    Min    : {np.min(dds_all):+.4f}")
    print(f"    Max    : {np.max(dds_all):+.4f}")


# ============================================================
# MAIN
# ============================================================

def run_pes_test_split():
    # ── 0. Setup resume environment ────────────────────────────
    setup_resume_environment(INPUT_CHECKPOINT_PATH, PES_SAVE_DIR)

    # ── 1. Load model ──────────────────────────────────────────
    print("\nLoading DDNN model...")
    model = DDNN(in_channels=1, growth_rate=6, block_config=[2, 4, 6, 4],
                 reduction=0.5, num_classes=1).to(DEVICE)
    checkpoint = torch.load(MODEL_PATH, map_location=DEVICE, weights_only=False)
    model.load_state_dict(checkpoint['model'])
    model.eval()
    print(f"Model loaded on {DEVICE}.\n")

    target_layer = model.blocks[-1].layers[-1][-1]
    gradcam      = GradCAM1D(model, target_layer)

    # ── 2. Build test-split index list ─────────────────────────
    with open(SPLIT_FILE, 'r') as f:
        split = json.load(f)
    test_patients = set(split['test_patients'])

    with h5py.File(H5_FILE, 'r') as f:
        all_groups = [g.decode('utf-8') if isinstance(g, bytes) else g
                      for g in f['groups'][:]]
        all_labels = f['y'][:]

    test_indices = [
        (i, int(all_labels[i]), all_groups[i])
        for i, g in enumerate(all_groups)
        if g in test_patients
    ]
    n_segs = len(test_indices)
    print(f"Test segments (raw)    : {n_segs:,}")
    print(f"Lead-agnostic total    : {n_segs * 2:,}  (this is what gets evaluated)\n")

    # ── 3. Resume from checkpoint if available ─────────────────
    results   = []
    start_seg = 0

    if os.path.exists(CHECKPOINT_FILE):
        ckpt      = np.load(CHECKPOINT_FILE, allow_pickle=True)
        results   = list(ckpt['results'])
        start_seg = int(ckpt['next_seg'])
        print(f"Resuming from segment {start_seg:,}  "
              f"({len(results):,} results already loaded)\n")
    else:
        print("No checkpoint found — starting fresh\n")

    # ── 4. Speed benchmark ─────────────────────────────────────
    if start_seg == 0:
        print("Benchmarking on first 20 segments...")
        t_bench = time.time()
        with h5py.File(H5_FILE, 'r') as hf:
            for i in range(min(20, n_segs)):
                seg_idx, _, _ = test_indices[i]
                X = hf['X'][seg_idx].astype(np.float32)
                for lead_num in range(2):
                    compute_dds_batched(model, gradcam, X[:, lead_num:lead_num+1])
        bench_elapsed   = time.time() - t_bench
        ms_per_eval     = (bench_elapsed / (min(20, n_segs) * 2)) * 1000
        remaining_evals = (n_segs - start_seg) * 2
        eta_sec         = remaining_evals * ms_per_eval / 1000
        print(f"Speed  : {ms_per_eval:.1f} ms per segment-lead")
        print(f"ETA    : {eta_sec/3600:.2f} hours  ({eta_sec/60:.0f} min)\n")

    # ── 5. Main evaluation loop ────────────────────────────────
    t0 = time.time()

    with h5py.File(H5_FILE, 'r') as hf:
        pbar = tqdm(
            test_indices[start_seg:],
            desc="Evaluating",
            initial=start_seg,
            total=n_segs,
            unit="seg"
        )

        for local_num, (seg_idx, label, group) in enumerate(pbar):
            global_num = start_seg + local_num
            source     = "SHDB" if group.startswith("SHDB") else "IRIDIA"
            X          = hf['X'][seg_idx].astype(np.float32)   # (3750, 2)

            for lead_num in range(2):
                dds_norm = compute_dds_batched(
                    model, gradcam, X[:, lead_num:lead_num+1]
                )
                results.append((seg_idx, lead_num, source, label, dds_norm))

            # ── Checkpoint (saves to /kaggle/working/ — download this!) ──
            if (global_num + 1) % CHECKPOINT_EVERY == 0:
                np.savez_compressed(
                    CHECKPOINT_FILE,
                    results  = np.array(results, dtype=object),
                    next_seg = global_num + 1
                )
                elapsed = time.time() - t0
                done    = (global_num + 1 - start_seg) * 2
                total   = (n_segs - start_seg) * 2
                rate    = done / elapsed if elapsed > 0 else 1
                remain  = (total - done) / rate
                pbar.set_postfix({
                    'saved_at': f'#{global_num+1:,}',
                    'eta':      f'{remain/60:.0f}m'
                })

    # ── 6. Save final results ──────────────────────────────────
    results_arr = np.array(results, dtype=object)
    np.savez_compressed(RESULTS_FILE, results=results_arr)
    print(f"\nFinal results saved → {RESULTS_FILE}")

    # Clean up checkpoint (run is complete)
    if os.path.exists(CHECKPOINT_FILE):
        os.remove(CHECKPOINT_FILE)
        print("Checkpoint removed (run complete)")

    # ── 7. PES summary ────────────────────────────────────────
    print_pes_summary(results_arr)

    print(f"\nTotal runtime: {(time.time() - t0)/3600:.2f} hours")


# ── If results already exist, just reprint the table ──────────
def load_and_print_pes():
    if not os.path.exists(RESULTS_FILE):
        print(f"Results file not found: {RESULTS_FILE}")
        return
    data = np.load(RESULTS_FILE, allow_pickle=True)
    print_pes_summary(data['results'])


if __name__ == "__main__":
    run_pes_test_split()